# **Introduction**

This notebook presents a modular, documented pipeline for quantitative **microglial** morphometric analysis.  
Although the workflow is technically compatible with any single‑cell dataset, it has been specifically optimized and documented for **microglia**, the resident immune cells of the central nervous system.  

Microglia exhibit a rich repertoire of morphological states—ranging from highly ramified homeostatic forms to compact ameboid phenotypes during activation.  
Because morphology is tightly linked to function, a robust and transparent morphometric pipeline is essential for studying microglial surveillance, activation, priming, phagocytosis, and resolution.

Emphasis is placed on **modularity** and **robustness** across heterogeneous datasets, imaging conditions, and activation states.

The analytical workflow is organized into clearly defined modules, each responsible for a specific stage of processing.  
This modular structure enables users to understand, modify, or extend individual components without disrupting the integrity of the entire pipeline.

---

### **Notebook Structure**

The notebook is organized into a sequence of self‑contained modules, each performing a distinct analytical task.  
To support both real and synthetic datasets, the workflow includes an optional preliminary module:

**Module -1 — Project Setup and Overview**  
Initial configuration of the environment, imports, and description of the workflow.

**Module 0 — Optional Synthetic Image Generation for High‑Resolution Microglial Morphometry**  
Provides a controlled, high‑quality synthetic dataset representing several canonical microglial activation states (resting, primed, activated, ameboid, resolving).  
These synthetic images serve as a pilot dataset for validating the entire morphometric pipeline.  
Researchers may freely substitute or complement them with their own experimental microglia.

**Module 1 — Dataset Loader**  
Automatically scans a user‑defined directory and loads all microglial images it contains.  
This module is fully dataset‑independent and supports any number of images.

**Module 2 — Core Image Processing Functions**  
Implements the essential preprocessing steps, including loading, thresholding, segmentation, skeletonization, bounding‑box detection, and optional scale normalization.

**Module 3 — Microglia preprocessing**  
Computes the fractal dimension of each microglial arbor, capturing global complexity and branching richness.

**Module 4 — Lacunarity Curves**  
Quantifies structural heterogeneity and gap distribution across spatial scales—particularly relevant for distinguishing ramified vs. compact microglial states.

**Module 5 — Gray‑level Co‑occurrence Matrix (GLCM) - Based Texture Metrics**  
Extracts gray‑level co‑occurrence matrix (GLCM) features to characterize local texture patterns, which often change during microglial activation.

**Module 6 — Multiscale Fractal Spectrum (MFS)**  
Computes MFS as the scale-dependent profile of local box-counting slopes; this is an operational multiscale descriptor, not a conventional multifractal spectrum.

**Module 7 — Graph‑Based Morphometrics**  
Converts microglial skeletons into graphs and extracts topological descriptors such as node degree, branching patterns, connectivity, and arbor geometry.

**Module 8 — Batch Sholl Analysis**  
Performs radial intersection analysis to quantify arbor complexity and spatial organization—an essential descriptor of microglial ramification.

**Module 9 — Statistical Testing**  
Evaluates group differences between microglial activation states using appropriate parametric or non‑parametric tests, including effect size estimation and FDR correction.

**Module 10 — Dimensionality Reduction & Feature Importance**  
Applies PCA and UMAP for global and nonlinear structure discovery, and uses Random Forest and SHAP to identify the most discriminative morphometric features across microglial states.

---

### **Why This Notebook Is Universal**

This notebook is intentionally designed to be:

**1. Dataset‑independent (within microglial imaging)**  
The pipeline does not assume any specific number of images, naming conventions, or activation states.  
All modules automatically adapt to the dataset size and/or groups.

**2. Modular and Extensible**  
Each analytical step is isolated in its own module.  
Users may replace or extend individual components without affecting the rest of the workflow.

**3. Reproducible**  
Parameters and random seeds are explicit. The scientific package versions are fixed, and output metadata record the environment; numerical behavior still needs verification on the target platform.

**4. Transparent**  
Every transformation is visualized and documented.  
Researchers can verify each stage before proceeding to quantitative analysis.

---

### **Important Notes for Users**

To ensure smooth execution:

1. **Run the modules in order.**  
   Later modules depend on functions and variables defined earlier.

2. **Dataset path.**
   Set DATASET_DIR in the notebook, or MICROGLIA_DATASET_DIR in the environment; otherwise a local folder dialog is used. A missing/unavailable selection stops execution with an explicit error.

3. **Ensure the dataset folder contains only images.**  
   Non‑image files may cause loading errors.

4. **Visual modules scale automatically.**  
   All plotting functions adapt dynamically to the number of images.

5. **Small or homogeneous datasets may trigger statistical warnings.**  
   Inspect these warnings: some tests may be unestimable. Missing statistical results are not evidence of no group difference.

6. **CSV files use dot decimals and comma separators.**  
   All `.csv` files generated by the application use the standard scientific format: dot as the decimal separator (`.`) and comma as the field separator (`,`).  
   Depending on your regional settings, spreadsheet software such as Excel may display some numeric values incorrectly, especially very small p-values written in scientific notation.  
   For reliable visualization in Excel, we recommend opening the corresponding `.xlsx` files when available, or importing the `.csv` manually and selecting comma as the delimiter and dot as the decimal separator.
---

### **Goal of This Notebook**

The objective of this notebook is to provide a reproducible, transparent, and universally applicable **microglial morphometric analysis framework** that can be used by any researcher, regardless of dataset size, activation state, or computational expertise.  
Its modular design ensures that the workflow can be easily adapted to new datasets, extended with additional metrics, or integrated into larger analytical frameworks.

As a practical demonstration, the notebook includes a synthetic microglia generator (Module 0) that produces several canonical activation states.  
These synthetic images serve as a pilot dataset for executing the entire morphometric pipeline end‑to‑end.  
Although synthetic microglia are used here as an example, the workflow remains fully compatible with any experimental dataset containing isolated microglial cells acquired at appropriate resolution.


### Version 1.1.0 — Scientific corrections

This release retains **Multiscale Fractal Spectrum (MFS)** as the operational name of the corrected local-slope profile. It renames junction pixels, removes the duplicate total_length predictor, counts Sholl crossings geometrically and exports separate, sphericity-corrected mixed-ANOVA effects. Unique relative-path identifiers prevent basename collisions.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">

## Module -1 — Project Setup and Overview

This release uses the same scientific algorithms as the corrected Windows, macOS and Linux build. Python 3.12 was used for validation. The initialization cell checks exact versions of the scientific dependencies.

By default, missing or different versions are installed with pip. To prohibit automatic installation, set the environment variable MICROGLIA_AUTO_INSTALL to 0 before running. If a changed package was already imported, restart the kernel and run the notebook from the beginning; installing a package does not replace modules already loaded in memory.

The initialization imports imaging, statistical and machine-learning libraries, reports versions, creates the exports and synthetic_microglia_40x folders, and defines the common plotting and CSV/XLSX export utilities. CSV uses comma separators and dot decimals; the accompanying XLSX preserves numeric values for spreadsheet users.

**Paths and execution:** In a notebook, the default base directory is the kernel's working directory. In the standalone script, it is the script directory. MICROGLIA_BASE_DIR overrides either location. No __file__ variable is required in a notebook.

Run cells in order in a fresh kernel. The companion .py contains the same code cells in the same order, including the scientific function definitions; no separate analysis_core.py file is required.

The synthetic generator remains available in Module 0. MICROGLIA_GENERATE_SYNTHETIC=0 skips its execution when using an existing dataset. Select a dataset explicitly as explained in Module 1. Tkinter is optional when a path is supplied programmatically.

### macOS dependency compatibility

On macOS (Intel and Apple Silicon), this release pins numba==0.62.1 and llvmlite==0.45.1, alongside numpy==2.3.5 and Python 3.12. Prebuilt wheels were verified for both architectures. The complete set of dependencies is resolved in one pip transaction, with binary-only installation for Numba and llvmlite, so UMAP/SHAP cannot first select newer versions requiring a local LLVM build. Other platforms retain their existing dependency selection. No scientific descriptor formula is changed by this packaging fix; native execution still requires verification on the target system.


In [ ]:
# ============================================================
# Scientific environment — validated versions (Python 3.12)
# ============================================================
import platform
import os
import sys
import glob
import math
import random
import warnings
import subprocess
import re
import json
from collections import defaultdict
from importlib.metadata import version as get_installed_version, PackageNotFoundError
from importlib.metadata import version as package_version
try:
    import tkinter as tk
    from tkinter import filedialog
except ImportError:
    tk = None
    filedialog = None

# Set MICROGLIA_AUTO_INSTALL=0 to prohibit automatic dependency installation.
AUTO_INSTALL_DEPENDENCIES = os.environ.get("MICROGLIA_AUTO_INSTALL", "1") == "1"
VALIDATED_PACKAGES = {'pillow': '13.0.0', 'numpy': '2.3.5', 'pandas': '3.1.1', 'matplotlib': '3.10.8', 'seaborn': '0.13.2', 'networkx': '3.7', 'pingouin': '0.7.0', 'umap-learn': '0.6.0', 'scikit-posthocs': '0.18.0', 'scikit-image': '0.26.0', 'opencv-python': '5.0.0.93', 'scipy': '1.17.0', 'scikit-learn': '1.8.0', 'statsmodels': '0.15.0', 'openpyxl': '3.1.5', 'shap': '0.54.1'}
PACKAGE_MODULES = {'pillow': 'PIL', 'scikit-learn': 'sklearn', 'scikit-image': 'skimage', 'opencv-python': 'cv2', 'scikit-posthocs': 'scikit_posthocs', 'umap-learn': 'umap'}

if sys.version_info < (3, 12):
    raise RuntimeError("This validated release requires Python 3.12 or newer; Python 3.12 was tested.")

# macOS Intel and Apple Silicon: use compatible prebuilt Numba/LLVM wheels.
# Resolve the complete pinned environment in ONE pip transaction so UMAP/SHAP
# cannot first pull a newer llvmlite that would require compilation from source.
if platform.system() == "Darwin":
    VALIDATED_PACKAGES.update({"numba": "0.62.1", "llvmlite": "0.45.1"})

restart_required = []
mismatched_packages = []
for package, required in VALIDATED_PACKAGES.items():
    try:
        installed = get_installed_version(package)
    except PackageNotFoundError:
        installed = None
    if installed != required:
        mismatched_packages.append(f"{package}=={required} (found {installed})")
        module_name = PACKAGE_MODULES.get(package, package.replace("-", "_"))
        if any(name == module_name or name.startswith(module_name + ".")
               for name in sys.modules):
            restart_required.append(package)
if mismatched_packages:
    if not AUTO_INSTALL_DEPENDENCIES:
        raise RuntimeError("Prepare the required environment: " + "; ".join(mismatched_packages))
    specs = [f"{package}=={required}" for package, required in VALIDATED_PACKAGES.items()]
    command = [sys.executable, "-m", "pip", "install"]
    if platform.system() == "Darwin":
        command.append("--only-binary=numba,llvmlite")
    print("[INFO] Installing the pinned scientific environment in one transaction.")
    subprocess.check_call(command + specs)
if restart_required:
    raise RuntimeError("Dependencies changed after import. Restart the kernel and run from the top: "
                       + ", ".join(restart_required))
print("[INFO] Scientific environment validated; Python:", platform.python_version())
OPENCV_AVAILABLE = True
SKIMAGE_AVAILABLE = True

# Imports 2
# ============================================================

import skimage
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import pingouin as pg
import umap.umap_ as umap
import shap

try:
    from PIL import Image
    PIL_AVAILABLE = True
except ImportError:
    PIL_AVAILABLE = False

from scipy.ndimage import convolve
from scipy.stats import f
from scipy.stats import (
    shapiro,
    levene,
    ttest_ind,
    f_oneway,
    mannwhitneyu,
    kruskal
)

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

from skimage import io, exposure
from skimage.color import rgb2gray
from skimage.draw import line
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import gaussian, threshold_otsu, threshold_local
from skimage.measure import label, regionprops
from skimage.morphology import skeletonize, remove_small_objects
from skimage.transform import resize
from skimage.util import img_as_ubyte

from statsmodels.stats.multitest import multipletests
from statsmodels.stats.diagnostic import lilliefors
from statsmodels.stats.multicomp import pairwise_tukeyhsd

import scikit_posthocs as sp

# ------------------------------------------------------------
# Environment info
# ------------------------------------------------------------
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", plt.matplotlib.__version__)
print("seaborn version:", sns.__version__)
print("scikit-image version:", skimage.__version__)
print("OpenCV version:", cv2.__version__)
print("Python executable:", sys.executable)

print("\n[INFO] ===== Starting pipeline =====")

# ------------------------------------------------------------
# Create exports folder
# ------------------------------------------------------------


# Get the directory where THIS script is located

def get_base_dir():
    if getattr(sys, "frozen", False):
        return Path(sys.executable).resolve().parent
    requested = os.environ.get("MICROGLIA_BASE_DIR")
    if requested:
        return Path(requested).expanduser().resolve()
    if "__file__" in globals():
        return Path(__file__).resolve().parent
    return Path.cwd()

SCRIPT_DIR = get_base_dir()

EXPORT_DIR = SCRIPT_DIR / "exports"
OUTPUT_DIR = SCRIPT_DIR / "synthetic_microglia_40x"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("[INFO] Script directory:", SCRIPT_DIR)
print("[INFO] EXPORT_DIR:", EXPORT_DIR)
print("[INFO] OUTPUT_DIR:", OUTPUT_DIR)

# ============================================================
# Universal table exporter
# ============================================================
# Every time the script writes a .csv file, it also writes a
# matching .xlsx file. The .xlsx file is recommended for Excel
# users because it avoids regional decimal/separator problems.
# ============================================================

def _safe_excel_sheet_name(name="Data"):
    """
    Excel sheet names cannot exceed 31 characters and cannot contain
    some special characters.
    """
    invalid_chars = ["\\", "/", "*", "[", "]", ":", "?"]
    name = str(name)

    for ch in invalid_chars:
        name = name.replace(ch, "_")

    if not name.strip():
        name = "Data"

    return name[:31]


def _format_excel_sheet(ws, df):
    """
    Basic formatting for Excel exports:
    - freeze header row
    - add autofilter
    - adjust column widths
    - use scientific notation for p-values
    - preserve numeric values as real numbers
    """
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    pvalue_like_columns = {
        "p_value",
        "p_adj",
        "p_adj_global",
        "p_adj_family",
        "levene_p", "p_unc", "p_GG", "p_HF", "sphericity_p"
    }

    for col_idx, col_name in enumerate(df.columns, start=1):
        col_letter = ws.cell(row=1, column=col_idx).column_letter
        col_name_str = str(col_name)
        col_name_lower = col_name_str.lower()

        # Estimate readable column width
        try:
            sample_values = df[col_name].astype(str).head(200).tolist()
            max_len = max([len(col_name_str)] + [len(v) for v in sample_values])
        except Exception:
            max_len = len(col_name_str)

        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 50)

        # Numeric formatting
        try:
            is_numeric = pd.api.types.is_numeric_dtype(df[col_name])
            is_integer = pd.api.types.is_integer_dtype(df[col_name])
            is_float = pd.api.types.is_float_dtype(df[col_name])
        except Exception:
            is_numeric = False
            is_integer = False
            is_float = False

        if is_numeric:
            if (
                col_name_lower in pvalue_like_columns
                or col_name_lower.endswith("_p")
                or "p_value" in col_name_lower
                or "p_adj" in col_name_lower
            ):
                number_format = "0.000E+00"
            elif is_integer:
                number_format = "0"
            elif is_float:
                number_format = "0.000000"
            else:
                number_format = "General"

            for row_idx in range(2, ws.max_row + 1):
                ws.cell(row=row_idx, column=col_idx).number_format = number_format


def _write_xlsx_copy_from_dataframe(df, csv_path):
    """
    Create an .xlsx copy next to a .csv file.
    """
    csv_path = Path(csv_path)
    xlsx_path = csv_path.with_suffix(".xlsx")

    try:
        sheet_name = _safe_excel_sheet_name(csv_path.stem)

        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
            df.to_excel(
                writer,
                sheet_name=sheet_name,
                index=False
            )

            ws = writer.sheets[sheet_name]
            _format_excel_sheet(ws, df)

        print(f"[INFO] Excel copy saved: {xlsx_path}")

    except Exception as e:
        print(f"[WARNING] Could not create Excel copy for {csv_path}: {e}")


# ------------------------------------------------------------
# Patch pandas DataFrame.to_csv once
# ------------------------------------------------------------
# This keeps all existing df.to_csv(...) calls working normally,
# but automatically creates a matching .xlsx file.
# ------------------------------------------------------------

if not hasattr(pd.DataFrame, "_original_to_csv_for_dual_export"):
    pd.DataFrame._original_to_csv_for_dual_export = pd.DataFrame.to_csv

    def _to_csv_and_xlsx(self, path_or_buf=None, *args, **kwargs):
        # First, write the normal CSV exactly as before
        result = pd.DataFrame._original_to_csv_for_dual_export(
            self,
            path_or_buf,
            *args,
            **kwargs
        )

        # Then, if the output is a CSV file path, create an XLSX copy
        if path_or_buf is not None:
            try:
                output_path = Path(path_or_buf)

                if output_path.suffix.lower() == ".csv":
                    _write_xlsx_copy_from_dataframe(self, output_path)

            except Exception as e:
                print(f"[WARNING] Automatic XLSX export failed for {path_or_buf}: {e}")

        return result

    pd.DataFrame.to_csv = _to_csv_and_xlsx

# Automatically save every figure generated in the notebook
plt.rcParams["savefig.directory"] = os.getcwd()
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["savefig.format"] = "png"

print(f"[INFO] OpenCV available: {OPENCV_AVAILABLE}")
print(f"[INFO] scikit-image available: {SKIMAGE_AVAILABLE}")
print(f"[INFO] PIL available: {PIL_AVAILABLE}")

# ============================================================
# Plotting guide
# ============================================================

def lighten_color(color, amount=0.6):
    """
    Lighten a matplotlib color.
    amount=0 -> original color
    amount=1 -> white
    """
    import colorsys
    import matplotlib.colors as mc

    try:
        c = mc.cnames[color]
    except Exception:
        c = color

    r, g, b = mc.to_rgb(c)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = 1 - amount * (1 - l)
    return colorsys.hls_to_rgb(h, l, s)


def should_use_boxplot(df, group_col="cell_state", image_threshold=5, group_threshold=5):
    """
    Switch from per-image bars to grouped boxplot + points when:
      - number of groups > group_threshold
      - OR any group has more than image_threshold images
    """
    if df.empty or group_col not in df.columns:
        return False

    n_groups = df[group_col].nunique(dropna=True)
    max_images_in_group = df[group_col].value_counts(dropna=True).max()

    return (n_groups > group_threshold) or (max_images_in_group > image_threshold)


def build_group_color_map(groups):
    return {
        state: plt.cm.tab10(i % 10)
        for i, state in enumerate(groups)
    }


def plot_bars_or_boxplot(
    df,
    value_col,
    group_col="cell_state",
    filename_col="sample_id",
    state_order=None,
    ylabel=None,
    xlabel=None,
    title=None,
    figsize_bar=(14, 6),
    figsize_box=(10, 6),
    save_path=None,
    image_threshold=5,
    group_threshold=5,
    show_legend=True
):
    """
    Universal plotting function:
      - small dataset -> bar plot per image
      - large dataset -> boxplot by group + individual points
    """
    df_plot = df.copy().dropna(subset=[value_col, group_col])

    if df_plot.empty:
        print(f"[WARNING] No valid data to plot for: {value_col}")
        return

    # Group ordering
    if state_order is None:
        groups = list(df_plot[group_col].dropna().unique())
    else:
        groups = [g for g in state_order if g in df_plot[group_col].dropna().unique()]
        groups += [g for g in df_plot[group_col].dropna().unique() if g not in groups]

    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=groups, ordered=True)

    sort_cols = [group_col]
    if filename_col in df_plot.columns:
        sort_cols.append(filename_col)

    df_plot = df_plot.sort_values(sort_cols).reset_index(drop=True)

    color_map = build_group_color_map(groups)
    use_box = should_use_boxplot(
        df_plot,
        group_col=group_col,
        image_threshold=image_threshold,
        group_threshold=group_threshold
    )

    # --------------------------------------------------------
    # CASE 1 — bar plot per image
    # --------------------------------------------------------
    if not use_box:
        df_plot["color"] = df_plot[group_col].astype(object).map(color_map)

        plt.figure(figsize=figsize_bar)

        plt.bar(
            x=np.arange(len(df_plot)),
            height=df_plot[value_col],
            color=df_plot["color"],
            edgecolor="black",
            linewidth=0.7
        )

        if filename_col in df_plot.columns:
            plt.xticks(
                ticks=np.arange(len(df_plot)),
                labels=df_plot[filename_col],
                rotation=75,
                ha="right",
                fontsize=8
            )
        else:
            plt.xticks(np.arange(len(df_plot)))

        plt.ylabel(ylabel if ylabel else value_col)
        plt.xlabel(xlabel if xlabel else "Image Filename")
        plt.title(title if title else value_col)

        if show_legend:
            handles = [
                plt.Line2D([0], [0], color=color_map[state], lw=8, label=state)
                for state in groups
            ]
            plt.legend(handles=handles, title="Group", bbox_to_anchor=(1.05, 1), loc="upper left")

        ax = plt.gca()
        ax.grid(axis="y", linestyle="--", alpha=0.3)

        plt.tight_layout()

        if save_path is not None:
            plt.savefig(save_path, dpi=600, bbox_inches="tight")

        plt.show()
        return

    # --------------------------------------------------------
    # CASE 2 — boxplot + individual points
    # --------------------------------------------------------
    plt.figure(figsize=figsize_box)

    light_palette = {g: lighten_color(color_map[g], 0.6) for g in groups}

    sns.boxplot(
        data=df_plot,
        x=group_col,
        y=value_col,
        hue=group_col,
        order=groups,
        palette=light_palette,
        dodge=False,
        width=0.55,
        showfliers=False,
        linewidth=1.2
    )

    sns.stripplot(
        data=df_plot,
        x=group_col,
        y=value_col,
        hue=group_col,
        order=groups,
        palette=color_map,
        dodge=False,
        jitter=0.12,
        alpha=0.9,
        size=5,
        edgecolor="black",
        linewidth=0.4
    )

    ax = plt.gca()
    legend = ax.get_legend()
    if legend is not None:
        legend.remove()

    ax.grid(axis="y", linestyle="--", alpha=0.3)

    plt.ylabel(ylabel if ylabel else value_col)
    plt.xlabel(xlabel if xlabel else "Group")
    plt.title(title if title else f"{value_col} by group")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=600, bbox_inches="tight")

    plt.show()

print("[INFO] ===== Environment Created =====")

### Synthetic Image Generation

Synthetic Image Generation

This module generates a small synthetic dataset of single-cell microglial images that can be used as a controlled reference set for testing and demonstrating the downstream morphometric workflow. Rather than replacing experimental microscopy data, it provides a reproducible collection of predefined microglial-like morphologies that facilitate pipeline development, visualization, and end-to-end benchmarking.

The generator creates 25 RGB images of isolated synthetic microglia (5 images for each of 5 predefined cell states) in a fixed 512 × 512 format. Each image contains a single centrally positioned cell rendered against a black background. The synthetic signal is distributed across biologically interpretable channels:

- Processes in the green channel
- Soma in the green channel
- Nucleus in the blue channel

To emulate a fluorescence/confocal-like appearance, the images are further processed with mild Gaussian blur and additive Gaussian noise.

The module simulates five predefined microglial morphological states:

- Resting microglia (small soma, numerous long and highly ramified processes)
- Primed microglia (slightly enlarged soma, fewer and less arborized branches)
- Activated microglia (enlarged soma with shorter, thicker processes)
- Ameboid microglia (large rounded soma with minimal branching)
- Resolving microglia (intermediate morphology with partial process re-extension)

These synthetic morphologies are generated through parameterized drawing rules that control soma radius, branch number, branch length, angular dispersion, thickness, and arborization probability. Reproducibility is enforced through a fixed random seed, allowing the same synthetic dataset to be regenerated consistently across runs and environments.

In addition to image generation, this module also:

- Saves all synthetic images as .png files in the output directory
- Creates a labels.csv file linking each filename to its corresponding microglial state
- Generates a 5 × 5 visualization panel summarizing the full synthetic dataset

Within the broader structure of the notebook, these synthetic images serve as a convenient demonstration dataset for running the subsequent stages of the analysis pipeline, including preprocessing, segmentation, morphometric extraction, visualization, and classification.

Although this synthetic dataset is primarily intended for testing and demonstration, the downstream workflow can also be applied to real experimental microglial images, provided that image quality and cell isolation are sufficient for reliable morphometric analysis.

In [ ]:
# ============================================================
# Synthetic Microglia Generator — 5 Functional States
# ============================================================
# Processes = GREEN channel
# Soma = GREEN
# Nucleus = BLUE
# ============================================================
print("[INFO] ===== Module 0 - Synthetic Microglia =====")
# ============================================================
# Global configuration
# ============================================================
N_IMAGES_TOTAL = 25
IMAGE_SIZE = (512, 512)
BACKGROUND_LEVEL = 0.0
MAX_INTENSITY = 1.0
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

CELL_STATES = [
    "resting_microglia",
    "primed_microglia",
    "activated_microglia",
    "ameboid_microglia",
    "resolving_microglia"
]

IMAGES_PER_STATE = N_IMAGES_TOTAL // len(CELL_STATES)

# ============================================================
# Utility functions
# ============================================================

def create_empty_image():
    """Return a pure black RGB image."""
    return np.zeros((IMAGE_SIZE[0], IMAGE_SIZE[1], 3), dtype=np.float32)

def add_green(img, rr, cc, intensity):
    """Add intensity to GREEN channel."""
    img[rr, cc, 1] = np.clip(img[rr, cc, 1] + intensity, 0, MAX_INTENSITY)
    return img

def add_blue(img, rr, cc, intensity):
    """Add intensity to BLUE channel."""
    img[rr, cc, 2] = np.clip(img[rr, cc, 2] + intensity, 0, MAX_INTENSITY)
    return img

def draw_soma(img, center, radius, intensity=0.8):
    """Draw soma in GREEN, nucleus in BLUE."""
    yy, xx = np.ogrid[:IMAGE_SIZE[0], :IMAGE_SIZE[1]]
    mask = (yy - center[0])**2 + (xx - center[1])**2 <= radius**2

    # Soma (green)
    img[mask, 1] = np.clip(img[mask, 1] + intensity, 0, MAX_INTENSITY)

    # Nucleus (blue)
    nucleus_mask = (yy - center[0])**2 + (xx - center[1])**2 <= (radius * 0.45)**2
    img[nucleus_mask, 2] = np.clip(img[nucleus_mask, 2] + 0.9, 0, MAX_INTENSITY)

    return img

def draw_branch(img, start, length, n_segments, angle_spread, base_angle,
                thickness=1, intensity=0.7, arborization_prob=0.0):

    y, x = start
    seg_length = length / n_segments
    current_angle = base_angle

    for _ in range(n_segments):
        current_angle += np.random.uniform(-angle_spread, angle_spread)

        dy = seg_length * np.sin(current_angle)
        dx = seg_length * np.cos(current_angle)

        new_y = int(y + dy)
        new_x = int(x + dx)

        rr, cc = line(int(y), int(x), new_y, new_x)

        # Processes in GREEN
        for t in range(-thickness, thickness + 1):
            rr_shift = np.clip(rr + t, 0, IMAGE_SIZE[0] - 1)
            cc_shift = np.clip(cc + t, 0, IMAGE_SIZE[1] - 1)
            img = add_green(img, rr_shift, cc_shift, intensity)

        # Arborization
        if np.random.rand() < arborization_prob:
            side_angle = current_angle + np.random.uniform(-np.pi/3, np.pi/3)
            side_len = length * np.random.uniform(0.1, 0.3)

            img, _ = draw_branch(
                img,
                start=(new_y, new_x),
                length=side_len,
                n_segments=max(2, n_segments // 3),
                angle_spread=angle_spread,
                base_angle=side_angle,
                thickness=max(1, thickness - 1),
                intensity=intensity * 0.8,
                arborization_prob=arborization_prob * 0.7
            )

        y, x = new_y, new_x

    return img, (y, x)

def add_gaussian_noise(img, sigma=0.02):
    noise = np.random.normal(0, sigma, img.shape)
    img = img + noise
    return np.clip(img, 0, MAX_INTENSITY)

def apply_confocal_blur(img, sigma=1.0):
    """Blur all channels slightly for realism."""
    blurred = img.copy()
    for c in range(3):
        blurred[:, :, c] = gaussian(img[:, :, c], sigma=sigma, preserve_range=True)
    return blurred

# ============================================================
# Microglia Morphology Generators
# ============================================================

def generate_resting_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=10, intensity=0.8)

    n_branches = np.random.randint(8, 12)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(120, 200),
            n_segments=12,
            angle_spread=np.deg2rad(8),
            base_angle=angle,
            thickness=1,
            intensity=0.6,
            arborization_prob=0.5
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.0), sigma=0.03)

def generate_primed_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=12, intensity=0.85)

    n_branches = np.random.randint(6, 9)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(100, 150),
            n_segments=10,
            angle_spread=np.deg2rad(12),
            base_angle=angle,
            thickness=1,
            intensity=0.7,
            arborization_prob=0.25
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.1), sigma=0.035)

def generate_activated_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=14, intensity=0.9)

    n_branches = np.random.randint(3, 6)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(40, 80),
            n_segments=5,
            angle_spread=np.deg2rad(20),
            base_angle=angle,
            thickness=2,
            intensity=0.8,
            arborization_prob=0.1
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.2), sigma=0.04)

def generate_ameboid_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=18, intensity=1.0)

    for _ in range(np.random.randint(0, 3)):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(20, 40),
            n_segments=3,
            angle_spread=np.deg2rad(25),
            base_angle=angle,
            thickness=3,
            intensity=0.9,
            arborization_prob=0.0
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.3), sigma=0.05)

def generate_resolving_microglia():
    img = create_empty_image()
    center = (256, 256)
    img = draw_soma(img, center, radius=12, intensity=0.85)

    n_branches = np.random.randint(4, 7)
    for _ in range(n_branches):
        angle = np.random.uniform(0, 2*np.pi)
        img, _ = draw_branch(
            img, center,
            length=np.random.uniform(80, 120),
            n_segments=8,
            angle_spread=np.deg2rad(15),
            base_angle=angle,
            thickness=1,
            intensity=0.7,
            arborization_prob=0.2
        )

    return add_gaussian_noise(apply_confocal_blur(img, sigma=1.1), sigma=0.035)

# ============================================================
# Main generation loop
# ============================================================

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print("[INFO] OUTPUT_DIR:", OUTPUT_DIR)
    print("[INFO] OUTPUT_DIR abs:", OUTPUT_DIR.resolve())

    generators = {
        "resting_microglia": generate_resting_microglia,
        "primed_microglia": generate_primed_microglia,
        "activated_microglia": generate_activated_microglia,
        "ameboid_microglia": generate_ameboid_microglia,
        "resolving_microglia": generate_resolving_microglia
    }

    records = []
    counter = 0

    for cell_state in CELL_STATES:
        for _ in range(IMAGES_PER_STATE):
            img = generators[cell_state]()
            fname = f"{cell_state}_{counter:03d}.png"
            save_path = OUTPUT_DIR / fname

            plt.imsave(save_path, img)
            
            records.append({"filename": fname, "cell_state": cell_state})
            counter += 1

    labels_path = OUTPUT_DIR / "labels.csv"
    pd.DataFrame(records).to_csv(labels_path, index=False)

# ============================================================
# Visualization panel
# ============================================================

def visualize_panel():
    files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.lower().endswith(".png")])[:25]

    fig, axes = plt.subplots(5, 5, figsize=(15, 15))

    for ax, fname in zip(axes.flatten(), files):
        img = plt.imread(OUTPUT_DIR / fname)
        ax.imshow(img)
        ax.set_title(fname, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(EXPORT_DIR / "Module 0 - synthetic_microglia_panel.png", dpi=600, bbox_inches="tight")
    plt.show()

if __name__ == "__main__" and os.environ.get("MICROGLIA_GENERATE_SYNTHETIC", "1") == "1":
    main()
    visualize_panel()

print("\n[INFO] ===== Module 0 Completed =====")

**Figure 1. Synthetic microglial dataset representing five canonical morphological states.**  
This panel illustrates the full set of 25 synthetic microglial images generated at 40× confocal‑like resolution, encompassing resting, primed, activated, ameboid, and resolving phenotypes. Each state exhibits characteristic morphological signatures: resting microglia display highly ramified arbors with fine distal processes; primed cells show moderate retraction and thickening of proximal branches; activated microglia exhibit short, thickened processes and enlarged somata; ameboid cells adopt a compact, minimally branched morphology; and resolving microglia partially re‑extend processes following activation. These controlled, state‑specific morphologies provide a standardized reference dataset for validating the morphometric pipeline, enabling systematic benchmarking of fractal, topological, and texture‑based descriptors under well‑defined structural conditions.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 1 — Dataset Loader

The loader recursively scans the selected directory for PNG, JPG/JPEG and TIF/TIFF files. With subfolders, the first folder below the dataset root defines cell_state. For files directly in the root, the group is derived from the filename prefix before the final underscore.

Dataset selection follows this order:

1. MICROGLIA_DATASET_DIR, if set.
2. A DATASET_DIR variable assigned in the notebook before this cell.
3. A local Tkinter folder dialog.

If the dialog is unavailable or cancelled, execution stops with an explicit message. The former placeholder path is no longer silently substituted. An empty dataset also stops with a clear error.

The loader returns image paths, groups, display basenames and **unique relative paths**. In the preprocessing cache and feature tables, filename and sample_id use the relative path, including subfolders. Thus group_A/cell.png and group_B/cell.png remain different subjects; identical basenames in nested folders of the same group also remain distinct. A short display_name is retained in the preprocessing cache.

This identifier is used consistently in merges and becomes cell_id in the master dataframe. Group labels are user-supplied metadata inferred from folder/filename organization; the software does not infer a biological activation state from morphology.


In [ ]:
# ============================================================
# Universal Multiclass Dataset Loader
# ============================================================

print("\n[INFO] ===== Module 1 - Dataset Loader =====")

# ------------------------------------------------------------
# Folder selector
# ------------------------------------------------------------
def select_dataset_directory():
    requested = os.environ.get("MICROGLIA_DATASET_DIR") or globals().get("DATASET_DIR")
    if requested:
        folder = Path(requested).expanduser().resolve()
        if not folder.is_dir():
            raise ValueError(f"Dataset directory does not exist: {folder}")
        return str(folder)
    if tk is None or filedialog is None:
        raise RuntimeError("Set DATASET_DIR or MICROGLIA_DATASET_DIR when Tkinter is unavailable.")
    root = None
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        root.update()
        selected = filedialog.askdirectory(title="Select microglia dataset folder", mustexist=True)
    except Exception as exc:
        raise RuntimeError("Folder dialog unavailable. Set DATASET_DIR or MICROGLIA_DATASET_DIR.") from exc
    finally:
        if root is not None:
            root.destroy()
    if not selected:
        raise ValueError("No dataset folder selected.")
    return selected

# ------------------------------------------------------------
# Dataset loader
# ------------------------------------------------------------
def load_dataset_multiclass(base_dir, extensions=("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff")):
    base_dir = os.path.abspath(base_dir)

    image_paths = []
    cell_states = []
    display_names = []
    unique_ids = []

    # Recursive search: supports images in subfolders and sub‑subfolders
    for ext in extensions:
        pattern = os.path.join(base_dir, "**", ext)
        for img in sorted(glob.glob(pattern, recursive=True)):
            img_abs = os.path.abspath(img)
            rel_path = os.path.relpath(img_abs, base_dir).replace("\\", "/")
            parts = Path(rel_path).parts

            # If folders are present, the group will be the first folder
            if len(parts) >= 2:
                state_name = parts[0]
            else:
                fname = os.path.basename(img_abs)
                state_name = os.path.splitext(fname)[0].rsplit("_", 1)[0]

            image_paths.append(img_abs)
            cell_states.append(state_name)
            display_names.append(os.path.basename(img_abs))  # Show Short name
            unique_ids.append(rel_path)                      # Unique identifier

    if len(image_paths) == 0:
        print("[WARNING] No images found.")
        return [], [], [], [], []

    classes = list(dict.fromkeys(cell_states))

    print(f"[INFO] Detected groups: {classes}")
    print(f"[INFO] Loaded {len(image_paths)} images from: {base_dir}")

    return image_paths, cell_states, classes, display_names, unique_ids
# ============================================================
# Select folder and load dataset
# ============================================================

DATASET_DIR = select_dataset_directory()

all_paths, all_cell_states, class_names, all_display_names, all_unique_ids = load_dataset_multiclass(DATASET_DIR)
if not all_paths:
    raise ValueError("No supported image files found in the selected dataset.")
state_order = class_names

print("\n[INFO] ===== Module 1 Completed =====")

<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## **Module 2 — Core Image Processing Functions**

This module defines foundational image processing utilities used in later stages of the microglial morphometric analysis pipeline.
At this stage, the focus is not on a full preprocessing workflow, but on a specific operation required for structural analysis: the extraction of a smoothed skeleton suitable for fractal dimension estimation.

'compute_skeleton_for_fd(binary)'

This function generates a clean and permissive skeleton representation from a binary image.
It is specifically designed for fractal dimension (FD) analysis, where preserving the overall structure of the signal is more important than aggressively simplifying it.

The function expects a binary input image (foreground = microglial structure, background = 0) and applies the following steps:

1. Binary normalization

The input is converted to a boolean array to ensure consistent processing:

- Non-zero values → foreground (True)
- Zero values → background (False)

If the input contains no foreground pixels, the function returns an empty skeleton.

2. Removal of very small objects

Small connected components are removed using:

'remove_small_objects(max_size=29)'

This step eliminates tiny artifacts while preserving most of the microglial structure.
The threshold is intentionally permissive to avoid removing fine processes.

3. Mild smoothing

A Gaussian filter is applied:

- Sigma = 1.0
- Followed by thresholding (> 0.2)

This operation:

- reduces pixel-level noise
- smooths irregular edges
- improves skeleton continuity

4. Skeletonization

The processed binary image is reduced to a 1-pixel-wide skeleton using:

'skimage.morphology.skeletonize'

This skeleton preserves the global topology of the structure and is suitable for:

- Fractal dimension estimation
- Structural complexity analysis
- Downstream graph-based representations

---

### **Design Philosophy**

This function is intentionally designed to be:

- Permissive — preserves as much structural information as possible
- Minimalistic — avoids aggressive filtering or pruning
- Deterministic — produces consistent outputs for identical inputs
- FD-oriented — optimized for fractal analysis rather than visual cleanliness

Notably, the function does not perform:

- Largest connected component selection
- Branch pruning or spur removal
- Morphological simplification beyond basic smoothing

These choices ensure that fine microglial processes are retained, which is critical for accurate fractal measurements.

---

### **Role in the Pipeline**

The output of this function serves as a structural representation of microglial morphology and is primarily used in:

- Fractal dimension computation
- Structural complexity metrics
- Graph-based morphometric analysis

This module establishes an initial building block for more advanced processing steps defined in subsequent modules.

In [ ]:
# ============================================================
# Core Image Processing Functions
# ============================================================

print("\n[INFO] ===== Module 2 - Core Image Processing Functions =====")

# ------------------------------------------------------------
# Skeleton for fractal dimension
# ------------------------------------------------------------
def compute_skeleton_for_fd(binary):
    """
    Simple and permissive FD skeleton:
    - keep most foreground signal
    - mild smoothing
    - no largest-component filtering
    - no spur pruning
    """
    binary = binary.astype(bool)

    if not np.any(binary):
        return np.zeros_like(binary, dtype=bool)

    binary_clean = remove_small_objects(binary, max_size=29)
    binary_smooth = gaussian(binary_clean.astype(float), sigma=1.0) > 0.2
    skel = skeletonize(binary_smooth)
    return skel

print("\n[INFO] ===== Module 2 Completed =====")

### Notes on skeleton representations

Preprocessing retains two skeleton representations. Module 6 uses skeleton_fd_uint8 for the MFS profile. Graph morphometrics and Sholl both use skeleton_visual_uint8. The global box-counting FD in Module 3 is computed from the binary mask, not from a silently substituted graph skeleton.

These representations serve different computational purposes and should be inspected before interpretation. Skeletonization does not guarantee preservation of every process or a nonempty result. Segmentation, smoothing, thresholding and resizing can affect subsequent measurements. Quantities expressed in pixels refer to the preprocessed image grid, which is currently 1024 × 1024; they are not automatically calibrated in micrometres.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


### **Module 3 — Fractal Dimension (Box‑Counting Method)**

This module performs three key tasks within the microglial morphometric analysis pipeline:

- Loads and preprocesses microglial images,
- Generates an ordered visual inspection panel for quality control, and
- Computes fractal dimension (FD) using the classical box-counting method.

Rather than being limited to fractal analysis alone, this module establishes the processed image representations required for downstream structural quantification and provides both numerical and visual outputs for evaluating microglial morphology.

It is designed to work with both synthetic microglial images and real microscopy data, provided that the images can be loaded as grayscale-compatible inputs.

---

1. Image Loading
'load_image(path)'

This function loads an image from disk and converts it into a grayscale representation suitable for preprocessing.

The loader uses a hierarchical backend strategy:

- OpenCV, when available
- Scikit-image, as an alternative backend
- Pillow (PIL), as a secondary fallback

For multichannel images, the function converts the signal to grayscale before further processing.
In the OpenCV path, RGB-like images are converted using standard grayscale conversion.
In the scikit-image fallback, multichannel images are reduced using a weighted combination of the green and blue channels, which is appropriate for the synthetic microglial images generated earlier in the notebook.

If the image cannot be loaded by any backend, the function raises an explicit error.

2. Image Preprocessing
'preprocess_image(img, target_size=(1024, 1024), ...)'

This function standardizes each image prior to quantitative analysis.
The preprocessing workflow includes the following steps:

- Resizing

All images are resized to a fixed target resolution (default: 1024 × 1024 px), ensuring consistent scale for downstream measurements, uses adaptive local thresholding with a minimum-signal constraint, and keeps RGB/RGBA conversion to grayscale before segmentation. This improves preservation of fine microglial processes while keeping downstream metrics comparable across images.

- Optional Gaussian blur

A mild Gaussian blur can be applied before thresholding in order to reduce local noise and improve segmentation stability.

- Thresholding

The module supports two segmentation modes:

Otsu thresholding (default)
adaptive local thresholding (optional)

This step converts the grayscale image into a binary mask representing the segmented microglial structure.

- Small-component removal

Connected components below a minimum size threshold can be removed in order to suppress small artifacts.

- Small-hole filling

Small holes inside the segmented structure can be filled to produce more coherent binary objects.

- Skeleton generation

Two skeleton representations are generated:

visual skeleton — a standard skeleton used for inspection and display
FD skeleton — a smoothed and more permissive skeleton produced by compute_skeleton_for_fd, intended for fractal analysis
g. Bounding box extraction

A bounding box is computed from the binary foreground coordinates.
If no foreground is detected, the function returns a default bounding box covering the full target image.

The function returns:

- resized grayscale image
- binary mask
- visual skeleton
- FD-oriented skeleton
- bounding box

All image-like outputs are converted to uint8 for compatibility and consistency.

3. Ordered Visual Inspection Panel
'visualize_preprocessing_panel_ordered(...)'

To facilitate preprocessing quality control, the module creates a structured visualization panel showing the main processing stages for selected images.

For each displayed sample, the panel includes:

- original image
- resized image
- binary segmentation
- visual skeleton

The visualization is intentionally:

- grouped by cell state
- ordered according to the preferred class order
- limited to a maximum number of classes
- limited to a maximum number of images per class

This design makes it easier to inspect whether preprocessing behaves consistently across microglial phenotypes while keeping the notebook responsive for large datasets.

The resulting panel is exported to:

'Module 3 - processing.png'

4. Fractal Dimension (Box-Counting Method)
'fractal_dimension(binary_img)'

This function implements a classical and permissive box-counting algorithm for estimating the fractal dimension of a binary microglial structure.

The function expects a 2D binary image and proceeds as follows:

-  Boolean conversion

The input image is converted to boolean format.

-  Empty-mask handling

If no foreground pixels are present, the function returns:

- fd = 0.0
- empty lists for sizes and counts
- zero-valued coefficients

- Multiscale box counting

The image is partitioned into square boxes of decreasing size, obtained by repeated division by 2 from the maximum possible box size down to 2 pixels.

For each box size, the algorithm counts how many boxes contain at least one foreground pixel.

- Log-log regression

The fractal dimension is estimated as the slope of the linear fit between:

- log(1 / box_size)
- log(number_of_non_empty_boxes)

The function returns:

- fd (estimated fractal dimension)
- sizes (list of box sizes used)
- counts (number of non-empty boxes at each scale)
- coeffs (regression coefficients from the log-log fit)

This implementation is intentionally simple and permissive:

- no foreground cropping
- no extra filtering inside the FD function
- no morphology-specific assumptions beyond binarity

5. Preprocessing Cache and FD Summary Table

The module preprocesses all images listed in the dataset loader outputs and stores the results in a dictionary named preprocessed_data.

For each image, the cache stores:

- filename
- cell state
- resized image
- binary mask
- visual skeleton
- FD skeleton
- bounding box

The module then computes fractal dimension for each image and assembles a summary table containing:

- filename
- cell_state
- fractal_dimension

This table is sorted according to the preferred class order and exported as:

'Module 3 - fractal_dimension_summary_table.csv'

6. Visualization of Fractal Dimension Results

Finally, the module visualizes FD values across images or groups using the shared plotting utility plot_bars_or_boxplot(...).

Depending on dataset size and class structure, the plot is displayed either as:

- a bar plot per image, or
- a boxplot with individual data points by group

The resulting figure is exported as:

'Module 3 - bar_plot_of_fractal_dimension.png'

---

### **Role in the Pipeline**

This module is a bridge between raw input images and quantitative morphological analysis.
It standardizes the image representation, enables visual quality control, and computes one of the key global descriptors of microglial structural complexity: fractal dimension.

Its outputs support both immediate interpretation and later modules involving comparative morphometrics, statistical analysis, and classification.

In [ ]:
# ============================================================
# Microglia preprocessing + Visual Inspection Panel
# ============================================================

print("\n[INFO] ===== Module 3 - Microglia preprocessing =====")

# ============================================================
# Module 3 configuration
# ============================================================

# Recommended for high-resolution microglia images.
# Use (512, 512) if processing is too slow.
PROCESS_TARGET_SIZE = (1024, 1024)

# Adaptive threshold is better for fluorescence images with
# bright soma + faint processes + uneven background.
USE_ADAPTIVE_THRESHOLD = True

# En skimage threshold_local usa: threshold = local_mean - offset
# Por eso offset positivo puede hacer que el fondo negro entre como objeto.
# Usamos offset negativo para hacerlo más estricto.
ADAPTIVE_OFFSET = -5

# Evita que el fondo muy oscuro se segmente aunque el threshold local falle.
MIN_SIGNAL_INTENSITY = 10

# Object size must increase when image size increases.
# Suggested:
#   512  -> 80-150
#   1024 -> 200-400
MIN_OBJECT_SIZE = 250

# Fill only small holes inside the segmented cell.
MIN_HOLE_SIZE = 500

# Mild blur to reduce pixel noise before thresholding.
GAUSSIAN_BLUR = True

# Keep conservative fluorescence enhancement.
AUTO_ENHANCE = True

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def load_image(path):
    """
    Load image and convert RGB/RGBA to grayscale.

    This keeps your preferred behavior:
    RGB image -> grayscale image.

    The output is always uint8 in range 0-255.
    """

    # --- OpenCV backend ---
    if OPENCV_AVAILABLE:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

        if img is not None:
            # RGB/BGR/RGBA/BGRA -> grayscale
            if img.ndim == 3:
                if img.shape[2] == 4:
                    img = cv2.cvtColor(img, cv2.COLOR_BGRA2GRAY)
                else:
                    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            img = exposure.rescale_intensity(
                img.astype(np.float32),
                out_range=(0, 255)
            ).astype(np.uint8)

            return img

    # --- scikit-image backend ---
    try:
        img = io.imread(path)

        # RGB/RGBA -> grayscale
        if img.ndim == 3:
            if img.shape[2] == 4:
                img = img[:, :, :3]
            img = rgb2gray(img)

        img = img.astype(np.float32)

        if img.max() > 0:
            img = exposure.rescale_intensity(
                img,
                out_range=(0, 255)
            )

        return img.astype(np.uint8)

    except Exception:
        pass

    # --- PIL backend ---
    if PIL_AVAILABLE:
        try:
            img = Image.open(path).convert("L")
            img = np.array(img).astype(np.float32)

            if img.max() > 0:
                img = exposure.rescale_intensity(
                    img,
                    out_range=(0, 255)
                )

            return img.astype(np.uint8)

        except Exception:
            pass

    raise ValueError(f"[ERROR] Could not load image: {path}")

# ------------------------------------------------------------
# Preprocess
# ------------------------------------------------------------
def _remove_small_components(binary, min_size):
    labeled_img = label(binary)
    cleaned = np.zeros_like(binary, dtype=bool)

    for region in regionprops(labeled_img):
        if region.area >= min_size:
            cleaned[labeled_img == region.label] = True

    return cleaned

def auto_enhance_fluorescence(
    img,
    p_low=1,
    p_high=99.5,
    min_p95_intensity=80,
    max_gain=2.0
):
    """
    Conservative rescue of weak fluorescence.
    Only boosts clearly dim images and caps the gain.
    Avoids homogenizing biological differences.
    """

    img = img.astype(np.float32)

    if img.max() <= 1.0:
        img *= 255.0

    if img.max() == 0:
        return img.astype(np.uint8)

    p95 = np.percentile(img, 95)

    # Only rescue underexposed images
    if p95 < min_p95_intensity and p95 > 0:
        gain = min(min_p95_intensity / p95, max_gain)
        img = img * gain

    # Mild percentile clipping only
    low, high = np.percentile(img, (p_low, p_high))

    if high > low:
        img = exposure.rescale_intensity(
            img,
            in_range=(low, high),
            out_range=(0, 255)
        )

    return np.clip(img, 0, 255).astype(np.uint8)

def _fill_small_holes(binary, min_hole_size):
    inv = ~binary
    labeled_img = label(inv)
    filled = binary.copy()

    for region in regionprops(labeled_img):
        if region.area < min_hole_size:
            coords = region.coords
            filled[coords[:, 0], coords[:, 1]] = True

    return filled

def preprocess_image(
    img,
    target_size=PROCESS_TARGET_SIZE,
    use_adaptive_threshold=USE_ADAPTIVE_THRESHOLD,
    adaptive_offset=ADAPTIVE_OFFSET,
    min_signal_intensity=MIN_SIGNAL_INTENSITY,
    gaussian_blur=GAUSSIAN_BLUR,
    min_object_size=MIN_OBJECT_SIZE,
    min_hole_size=MIN_HOLE_SIZE,
    clean_holes=True,
    auto_enhance=AUTO_ENHANCE
):
    """
    Preprocess grayscale fluorescence image:
      1. Resize to common size for comparable metrics
      2. Enhance fluorescence conservatively
      3. Mild blur
      4. Adaptive or Otsu threshold
      5. Remove small objects
      6. Fill small holes
      7. Skeletonize
    """

    # --------------------------------------------------------
    # Resize to common size
    # --------------------------------------------------------
    if target_size is None:
        img_resized = img.copy()
    else:
        if OPENCV_AVAILABLE:
            # INTER_AREA is good for downsampling.
            # INTER_CUBIC is better if an image is smaller than target_size.
            if img.shape[0] > target_size[0] or img.shape[1] > target_size[1]:
                interpolation = cv2.INTER_AREA
            else:
                interpolation = cv2.INTER_CUBIC

            img_resized = cv2.resize(
                img,
                target_size,
                interpolation=interpolation
            )
        else:
            img_resized = resize(
                img,
                target_size,
                preserve_range=True,
                anti_aliasing=True
            ).astype(np.uint8)

    # --------------------------------------------------------
    # Automatic fluorescence enhancement
    # --------------------------------------------------------
    if auto_enhance:
        img_resized = auto_enhance_fluorescence(
            img_resized,
            p_low=1,
            p_high=99.8,
            min_p95_intensity=70,
            max_gain=1.5
        )

    # --------------------------------------------------------
    # Optional blur
    # --------------------------------------------------------
    if gaussian_blur:
        if OPENCV_AVAILABLE:
            img_blur = cv2.GaussianBlur(img_resized, (3, 3), 0)
        else:
            img_blur = gaussian(
                img_resized,
                sigma=0.5,
                preserve_range=True
            ).astype(np.uint8)
    else:
        img_blur = img_resized

    # --------------------------------------------------------
    # Thresholding
    # --------------------------------------------------------
    if use_adaptive_threshold:
        # Window size adapted to image size.
        # For 1024 px this gives ~71 px.
        # For 512 px this gives ~51 px.
        block_size = int(min(img_blur.shape) * 0.07)

        # threshold_local requires odd block size
        if block_size % 2 == 0:
            block_size += 1

        block_size = max(block_size, 51)

        adaptive_thresh = threshold_local(
            img_blur,
            block_size=block_size,
            method="gaussian",
            offset=adaptive_offset
        )

        # Two conditions:
        # 1. Pixel must be above local threshold.
        # 2. Pixel must have minimum real fluorescence signal.
        binary = (img_blur > adaptive_thresh) & (img_blur > min_signal_intensity)

    else:
        thresh = threshold_otsu(img_blur)

        # Also apply minimum signal filter for safety.
        binary = (img_blur > thresh) & (img_blur > min_signal_intensity)

    # --------------------------------------------------------
    # Remove small objects
    # --------------------------------------------------------
    if min_object_size > 0:
        binary = _remove_small_components(binary, min_object_size)

    # --------------------------------------------------------
    # Fill small holes
    # --------------------------------------------------------
    if clean_holes and min_hole_size > 0:
        binary = _fill_small_holes(binary, min_hole_size)

    # --------------------------------------------------------
    # Skeletons
    # --------------------------------------------------------
    skeleton_visual = skeletonize(binary)
    skeleton_fd = compute_skeleton_for_fd(binary)

    # --------------------------------------------------------
    # Convert to uint8
    # --------------------------------------------------------
    binary_uint8 = binary.astype(np.uint8) * 255
    skeleton_visual_uint8 = skeleton_visual.astype(np.uint8) * 255
    skeleton_fd_uint8 = skeleton_fd.astype(np.uint8) * 255

    # --------------------------------------------------------
    # Bounding box
    # --------------------------------------------------------
    coords = np.column_stack(np.where(binary))

    if len(coords) > 0:
        y_min, x_min = coords.min(axis=0)
        y_max, x_max = coords.max(axis=0)
        bbox = (y_min, x_min, y_max, x_max)
    else:
        bbox = (0, 0, img_resized.shape[0], img_resized.shape[1])

    return (
        img_resized,
        binary_uint8,
        skeleton_visual_uint8,
        skeleton_fd_uint8,
        bbox
    )

# ------------------------------------------------------------
# Visual inspection panel
# ------------------------------------------------------------
def visualize_preprocessing_panel_ordered(
    image_paths,
    cell_states,
    preprocessed_data,
    max_classes=5,
    max_images_per_class=10,
    preferred_order=None,
    save_path=None
):
    if save_path is None:
        save_path = os.path.join(EXPORT_DIR, "Module 3 - processing.png")

    class_to_images = defaultdict(list)
    for p, s in zip(image_paths, cell_states):
        class_to_images[s].append(p)

    classes = list(class_to_images.keys())

    if preferred_order is not None:
        ordered = [c for c in preferred_order if c in classes]
        ordered += [c for c in classes if c not in ordered]
    else:
        ordered = classes

    selected = []
    for cls in ordered[:max_classes]:
        for p in class_to_images[cls][:max_images_per_class]:
            selected.append((p, cls))

    fig, axes = plt.subplots(
        len(selected), 4,
        figsize=(12, 3 * len(selected)),
        squeeze=False
    )

    for i, (path, state) in enumerate(selected):
        try:
            data = preprocessed_data[path]

            # original en gris
            img_original = load_image(path)

            r = data["img_resized"]
            b = data["binary_uint8"]
            s = data["skeleton_visual_uint8"]

            axes[i, 0].imshow(img_as_ubyte(img_original), cmap="gray")
            axes[i, 0].set_title(f"{state}\n{data['sample_id']}", fontsize=8)
            axes[i, 0].axis("off")

            axes[i, 1].imshow(r, cmap="gray")
            axes[i, 1].set_title(f"Resized {PROCESS_TARGET_SIZE[0]}x{PROCESS_TARGET_SIZE[1]}")
            axes[i, 1].axis("off")

            axes[i, 2].imshow(b, cmap="gray")
            axes[i, 2].set_title("Binary")
            axes[i, 2].axis("off")

            axes[i, 3].imshow(s, cmap="gray")
            axes[i, 3].set_title("Skeleton")
            axes[i, 3].axis("off")

        except Exception as e:
            for j in range(4):
                axes[i, j].text(0.5, 0.5, str(e), ha="center", va="center")
                axes[i, j].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
    
# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
# ============================================================
# Preprocessing cache
# ============================================================

path_to_display = dict(zip(all_paths, all_display_names))
path_to_uid = dict(zip(all_paths, all_unique_ids))

preprocessed_data = {}

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        img = load_image(path)

        img_resized, binary_uint8, skeleton_visual_uint8, skeleton_fd_uint8, bbox = preprocess_image(
            img,
            target_size=PROCESS_TARGET_SIZE,
            use_adaptive_threshold=USE_ADAPTIVE_THRESHOLD,
            adaptive_offset=ADAPTIVE_OFFSET,
            min_signal_intensity=MIN_SIGNAL_INTENSITY,
            gaussian_blur=GAUSSIAN_BLUR,
            min_object_size=MIN_OBJECT_SIZE,
            min_hole_size=MIN_HOLE_SIZE,
            clean_holes=True,
            auto_enhance=AUTO_ENHANCE
        )

        preprocessed_data[path] = {
            "filename": path_to_uid[path],
            "display_name": path_to_display[path],
            "sample_id": path_to_uid[path],
            "cell_state": cell_state,
            "img_resized": img_resized,
            "binary_uint8": binary_uint8,
            "skeleton_visual_uint8": skeleton_visual_uint8,
            "skeleton_fd_uint8": skeleton_fd_uint8,
            "bbox": bbox
        }

    except Exception as e:
        print(f"[PREPROCESS ERROR] {path}: {e}")

visualize_preprocessing_panel_ordered(
    all_paths,
    all_cell_states,
    preprocessed_data=preprocessed_data,
    max_classes=5,
    max_images_per_class=10,
    preferred_order=state_order,
    save_path=os.path.join(EXPORT_DIR, "Module 3 - processing.png")
)

**Figure 2. End‑to‑end preprocessing workflow: from raw microglial imagery to skeletonized arbors.**  
This figure illustrates the complete preprocessing pipeline applied to individual microglial images, including the original grayscale input, Otsu‑based thresholding, binary mask generation, and final skeleton extraction using the Zhang–Suen thinning algorithm. Otsu’s adaptive thresholding reliably separates microglial structures from background intensity variations, producing clean binary masks suitable for downstream morphometrics. The resulting skeletons preserve the topological organization of the arbor—branch points, endpoints, and overall geometry—while removing redundant pixel thickness. This step is critical for fractal analysis, graph‑based topology, and Sholl profiling, as it provides a standardized, one‑pixel‑wide representation of microglial morphology that is robust across heterogeneous imaging conditions and activation states.


In [ ]:
# ============================================================
# Fractal Dimension (Box-Counting Method)
# ============================================================
# Universal, reproducible, and robust implementation.
#
# Key design principles:
#   - Works with ANY microglial activation state
#   - Uses boolean masks (binary or skeleton) safely
#   - Avoids superfluous outputs
#   - Handles empty or degenerate masks gracefully
#   - Uses powers of two for reproducibility across datasets
#
# Returned values:
#   fd       → estimated fractal dimension
#   sizes    → box sizes used
#   counts   → number of non-empty boxes at each scale
#   coeffs   → (slope, intercept) from log–log regression
#
# Researchers may plug ANY microglial binary structure here:
#   - binary mask
#   - FD skeleton
#   - thresholded channels
# ============================================================

def fractal_dimension(binary_img):
    """
    Classical and permissive box-counting FD.
    No foreground cropping, no extra filtering.
    """
    binary = binary_img.astype(bool)

    if not np.any(binary):
        return 0.0, [], [], (0.0, 0.0)

    h, w = binary.shape
    max_size = min(h, w)

    sizes = []
    size = max_size

    while size >= 2:
        sizes.append(size)
        size //= 2

    counts = []

    for size in sizes:
        n_h = h // size
        n_w = w // size

        count = 0
        for i in range(n_h):
            for j in range(n_w):
                block = binary[i * size:(i + 1) * size, j * size:(j + 1) * size]
                if np.any(block):
                    count += 1

        counts.append(count)

    sizes_log = np.log(1 / np.array(sizes))
    counts_log = np.log(np.array(counts))

    coeffs = np.polyfit(sizes_log, counts_log, 1)
    fd = coeffs[0]

    return fd, sizes, counts, coeffs

# ============================================================
# Fractal Dimension Summary Table
# ============================================================

preferred_order = state_order
fd_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        binary_uint8 = data["binary_uint8"]
        binary_bool = binary_uint8.astype(bool)

        if np.any(binary_bool):
            fd, _, _, _ = fractal_dimension(binary_bool)
        else:
            fd = None

        fd_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            "fractal_dimension": fd
        })

    except Exception as e:
        fd_rows.append({
            "filename": path_to_uid[path],
            "sample_id": path,
            "cell_state": cell_state,
            "fractal_dimension": None,
            "error": str(e)
        })

df_fd = pd.DataFrame(fd_rows)

if preferred_order is not None:
    df_fd["order_key"] = df_fd["cell_state"].apply(
        lambda x: state_order.index(x) if x in state_order else len(state_order)
    )
    df_fd = df_fd.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])
else:
    df_fd = df_fd.sort_values(by=["cell_state", "filename"])

df_fd = df_fd[["filename", "sample_id", "cell_state", "fractal_dimension"]]

# Uses dot decimal and comma separator.
df_fd.to_csv(os.path.join(EXPORT_DIR, "Module 3 - fractal_dimension_summary_table.csv"), index=False)

df_fd

# ------------------------------------------------------------
# Apply ordering if provided
# ------------------------------------------------------------
if preferred_order is not None:
    df_fd["order_key"] = df_fd["cell_state"].apply(
        lambda x: state_order.index(x) if x in state_order else len(state_order)
    )

    df_fd = df_fd.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])

else:
    df_fd = df_fd.sort_values(by=["cell_state", "filename"])

### **Fractal Dimension Summary Table**

This block computes the fractal dimension (FD) for all images in the dataset using the box-counting method applied to the binary mask generated during preprocessing.

For each image:

- the precomputed binary mask is converted to boolean format
- the fractal dimension is estimated using the fractal_dimension function
- if no foreground structure is present, the FD value is set to None

To maintain clarity and usability, the results are stored in a compact summary table containing:

- the image filename
- the microglial activation state (cell_state)
- the estimated fractal dimension

Although the underlying FD computation produces additional intermediate outputs (box sizes, counts, and regression coefficients), only the final FD value is retained in this table to provide a concise and interpretable overview of structural complexity.

The table is optionally ordered according to the predefined class order to ensure consistent grouping across microglial activation states.

All results are exported as a CSV file:

'Module 3 - fractal_dimension_summary_table.csv'

This summary table provides a clean and reproducible representation of morphological complexity across images and conditions, and can be directly used for downstream statistical analysis, visualization, or machine learning workflows.

In [ ]:
# ------------------------------------------------------------
# Final clean table
# ------------------------------------------------------------
df_fd = df_fd[["filename", "sample_id", "cell_state", "fractal_dimension"]]

# ------------------------------------------------------------
# EXPORT TABLE (Fractal Dimension Summary Table)
# ------------------------------------------------------------
df_fd.to_csv(os.path.join(EXPORT_DIR, "Module 3 - fractal_dimension_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_fd

**Table 1. Fractal dimension values across five canonical microglial states.**  
This table summarizes the box‑counting fractal dimension (FD) computed for each synthetic microglial image. FD captures global branching complexity and is highly sensitive to morphological state. Resting microglia exhibit the highest FD values (≈1.35–1.44), reflecting their dense, highly ramified arbors. Primed microglia show intermediate FD values (≈1.18–1.23), consistent with partial process retraction. Resolving microglia display moderately reduced complexity (≈0.91–1.02), aligning with their transitional morphology. Activated and ameboid microglia present the lowest FD values (≈0.72–0.93), indicative of compact, minimally branched structures. These results demonstrate that fractal dimension robustly discriminates microglial activation states and provides a quantitative, scale‑invariant descriptor of morphological complexity.


In [ ]:
# ============================================================
# Plot of Fractal Dimension
# ============================================================

plot_bars_or_boxplot(
    df=df_fd,
    value_col="fractal_dimension",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Fractal Dimension (FD)",
    xlabel="Image Filename / Group",
    title="Fractal Dimension Summary",
    save_path=os.path.join(EXPORT_DIR, "Module 3 - bar_plot_of_fractal_dimension.png")
)

print("\n[INFO] ===== Module 3 Completed =====")

**Figure 3. Fractal dimension across microglial activation states.**  
This bar plot summarizes the fractal dimension (FD) of all microglial images, ordered by activation state. A clear gradient of morphological complexity emerges: resting microglia exhibit the highest FD values, reflecting their dense, highly ramified arbors; primed cells show intermediate complexity consistent with partial process retraction; resolving microglia occupy a transitional range; and both activated and ameboid cells display markedly reduced FD values, indicative of compact, minimally branched morphologies. The monotonic decrease in FD from resting to ameboid states demonstrates the sensitivity of fractal analysis to structural remodeling and highlights FD as a robust, scale‑invariant descriptor capable of discriminating microglial phenotypes with high fidelity.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">

## **Module 4 — Lacunarity Analysis (Gliding‑Box Method)**

This module computes lacunarity (Λ) as a measure of spatial heterogeneity in microglial structures using the classical gliding-box method.
It operates on the binary masks generated during preprocessing and provides both multiscale lacunarity curves and a compact summary metric for each image.

The module is designed to integrate directly with the outputs of the preprocessing stage and can be applied to both synthetic and real microglial datasets.

---

1. Lacunarity Computation
'compute_lacunarity(binary_img, box_sizes=[2, 4, 8, 16, 32, 64, 128, 256])'

This function computes lacunarity across multiple spatial scales using a sliding-window (gliding-box) approach.

- a. Input handling

- The input image is converted to uint8 format
- The function expects a binary structure (typically the preprocessing mask)

If the input contains no foreground pixels:

- lacunarity values are returned as NaN
- no valid box sizes are reported

- b. Box sizes

A predefined list of box sizes is used:

[2, 4, 8, 16, 32, 64, 128, 256]
- Box sizes larger than the image dimensions are automatically skipped
- The list can be modified depending on resolution or biological scale

- c. Gliding-window sampling

For each valid box size r:
- an 𝑟 × 𝑟 window is slid across the image
- the sum of foreground pixels is computed for every position
- these values form the distribution St

- d. Lacunarity formula

Lacunarity is computed as:

Λ(r) = ( Var(St) / (Mean(St))^2 ) + 1

where St represents the distribution of window sums.
- High Λ → heterogeneous, irregular structures
- Low Λ → compact, homogeneous structures

The function returns:

- lac_values (lacunarity values per box size)
- valid_sizes (box sizes actually used)

2. Lacunarity Curves for All Images

For each image in the dataset:

- the binary mask from preprocessing is used
- lacunarity is computed across all valid scales
- a lacunarity curve (Λ vs. box size) is generated

All curves are plotted in a grid layout:

- one subplot per image
- labeled by filename
- automatically adjusted to dataset size

The resulting figure is exported as:

'Module 4 - lacunarity_curves_for_all_images.png'

3. Lacunarity Summary Table

For downstream analysis, the module computes a single summary metric per image:

- mean lacunarity across all valid box sizes
'(np.nanmean(lac_values))'

A summary table is constructed containing:

- filename
- cell_state
- lacunarity_mean

The table is:

- ordered according to the predefined class order (state_order)
- cleaned to remove auxiliary columns

and exported as:

'Module 4 - lacunarity_summary_table.csv'

4. Visualization of Lacunarity Summary

The distribution of mean lacunarity values is visualized using the shared plotting utility:

'plot_bars_or_boxplot(...)'

Depending on dataset size, the results are displayed as:

- per-image bar plots, or
- grouped boxplots with individual data points

The figure is exported as:

'Module 4 - bar_plot_of_mean_lacunarity_values.png'

---

### **Interpretation in Microglial Morphology**

Lacunarity complements fractal dimension by capturing spatial heterogeneity rather than global scaling behavior.

In microglial analysis:

- highly ramified structures often exhibit higher lacunarity
- compact or ameboid morphologies tend to show lower lacunarity
- intermediate states may display transitional patterns

These trends reflect differences in branch distribution, gaps, and local density, although exact values depend on preprocessing and dataset characteristics.

---

### **Role in the Pipeline**

This module extends the structural analysis introduced by fractal dimension by adding a multiscale heterogeneity descriptor.

Together, fractal dimension and lacunarity provide complementary information about:

global complexity (FD)
spatial distribution and irregularity (Λ)

enabling a more complete characterization of microglial morphology across activation states and experimental conditions.

In [ ]:
# ============================================================
# Lacunarity Analysis (Gliding-Box Method)
# ============================================================
#
# Key design principles:
#   - Works with any microglial activation state
#   - Accepts any binary structure (mask or skeleton)
#   - Avoids unnecessary outputs
#   - Handles empty or degenerate masks safely
#   - Uses deterministic box sizes for reproducibility
#
# Returned values:
#   lac_values  → lacunarity values for each valid box size
#   valid_sizes → box sizes actually used in the computation
#
# Researchers may freely modify `box_sizes` depending on resolution, magnification, or biological scale of interest.
# ============================================================

print("\n[INFO] ===== Module 4 - Lacunarity Curves =====")

def compute_lacunarity(binary_img, box_sizes=[4, 8, 16, 32, 64, 128, 256]):
    """
    Fast lacunarity using an integral image.

    This avoids the very slow nested Python loops over every window.
    Recommended when images are resized to 1024x1024.
    """

    binary = binary_img.astype(np.uint8)
    h, w = binary.shape

    if not np.any(binary):
        return [np.nan] * len(box_sizes), []

    lac_values = []
    valid_sizes = []

    # Integral image:
    # integral[y, x] contains the sum of pixels above and left of (y, x)
    integral = np.pad(binary, ((1, 0), (1, 0)), mode="constant")
    integral = integral.cumsum(axis=0).cumsum(axis=1)

    for r in box_sizes:
        if r > h or r > w:
            continue

        # Fast sum of every r x r window
        sums = (
            integral[r:, r:]
            - integral[:-r, r:]
            - integral[r:, :-r]
            + integral[:-r, :-r]
        ).astype(np.float64)

        sums = sums.ravel()

        mean = sums.mean()
        var = sums.var()

        if mean == 0:
            lac = np.nan
        else:
            lac = var / (mean ** 2) + 1

        lac_values.append(lac)
        valid_sizes.append(r)

    return lac_values, valid_sizes

# ============================================================
# Lacunarity Curves for All Images
# ============================================================
# This block computes and plots lacunarity curves (Λ vs. box size) for each microglial image in the dataset.
#
# Design principles:
#   - Universal: works with any microglial activation state
#   - Reproducible: deterministic ordering and processing
#   - Minimalistic: only essential information is shown
#   - Robust: handles empty masks and processing errors safely
#
# Each subplot shows:
#   - box sizes used
#   - corresponding lacunarity values
#   - filename for identification
#
# Lacunarity helps distinguish microglial states:
#   - Resting microglia → higher Λ (heterogeneous, ramified)
#   - Activated / ameboid → lower Λ (compact, homogeneous)
#   - Primed / resolving → intermediate Λ
# ============================================================

lac_rows = []

n = len(all_paths)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, (path, cell_state) in enumerate(zip(all_paths, all_cell_states)):

    try:
        data = preprocessed_data[path]

        binary_uint8 = data["binary_uint8"]
        binary_bool = binary_uint8.astype(bool)

        lac_values, sizes = compute_lacunarity(binary_bool)

        lac_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            "lacunarity_mean": np.nanmean(lac_values)
        })

        ax = axes[i]
        ax.plot(sizes, lac_values, marker="o")
        ax.set_title(data["sample_id"], fontsize=9)
        ax.set_xlabel("Box size (pixels)")
        ax.set_ylabel("Lacunarity (Λ)")

    except Exception as e:
        ax = axes[i]
        ax.text(0.5, 0.5, f"Error:\n{os.path.basename(path)}", ha="center")
        ax.axis("off")

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Lacunarity Curves for All Images)
# ------------------------------------------------------------

plt.savefig(os.path.join(EXPORT_DIR, "Module 4 - lacunarity_curves_for_all_images.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show() 

**Figure 4. Lacunarity profiles across microglial activation states.**  
This panel displays lacunarity (λ) as a function of box size for all microglial samples, revealing consistent state‑dependent differences in spatial heterogeneity. Lacunarity decreases monotonically with increasing box size across all cells, reflecting the expected reduction in gap sensitivity at larger spatial scales. Resting microglia exhibit the lowest lacunarity values, consistent with their dense, uniformly distributed branching patterns. Primed and resolving microglia show intermediate profiles, indicating partial structural heterogeneity associated with transitional morphologies. Activated and ameboid microglia display the highest lacunarity, reflecting large contiguous gaps and reduced arborization. Together, these curves demonstrate that lacunarity provides a sensitive multiscale descriptor of microglial structural organization, complementing fractal dimension by capturing differences in local irregularity and gap distribution across activation states.


In [ ]:
# ============================================================
# Lacunarity Summary Table
# ============================================================
# This table summarizes lacunarity for each microglial image using:
#   - filename
#   - group / microglial activation state
#   - mean lacunarity across valid box sizes
#
# The table is ordered according to a user-defined biological progression (state_order).
# ============================================================

df_lac = pd.DataFrame(lac_rows)

# ------------------------------------------------------------
# Apply ordering
# ------------------------------------------------------------
df_lac["order_key"] = df_lac["cell_state"].apply(
    lambda x: state_order.index(x) if x in state_order else len(state_order)
)

df_lac = df_lac.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])

# ------------------------------------------------------------
# Final clean table
# ------------------------------------------------------------
df_lac = df_lac[["filename", "sample_id", "cell_state", "lacunarity_mean"]]

# ------------------------------------------------------------
# EXPORT TABLE (Lacunarity Summary Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_lac.to_csv(os.path.join(EXPORT_DIR, "Module 4 - lacunarity_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_lac  

**Table 2. Mean lacunarity values across microglial activation states.**  
This table reports the mean lacunarity (λ̄) for each microglial image, capturing differences in spatial heterogeneity and gap distribution across morphological states. Resting microglia show the lowest lacunarity values (≈7–10), reflecting their dense, uniformly ramified arbors with minimal large‑scale gaps. Primed microglia exhibit slightly higher λ̄ (≈17–28), consistent with early process retraction and emerging structural irregularity. Resolving microglia display intermediate lacunarity (≈36–64), reflecting partial recovery of arborization following activation. Activated and ameboid microglia show the highest λ̄ values (≈61–131), indicative of compact morphologies with large contiguous voids and reduced branching. These results highlight lacunarity as a sensitive multiscale descriptor that complements fractal dimension by quantifying local heterogeneity and structural discontinuity across microglial states.


In [ ]:
# ============================================================
# Plot of Mean Lacunarity Values
# ============================================================
plot_bars_or_boxplot(
    df=df_lac,
    value_col="lacunarity_mean",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Mean Lacunarity (Λ)",
    xlabel="Image Filename / Group",
    title="Mean Lacunarity Across Groups",
    save_path=os.path.join(EXPORT_DIR, "Module 4 - bar_plot_of_mean_lacunarity_values.png")
)

print("\n[INFO] ===== Module 4 Completed =====")

**Figure 5. Mean lacunarity across microglial activation states.**  
This bar plot summarizes the mean lacunarity (λ̄) for all microglial samples, ordered by activation state. A clear stratification emerges: resting microglia exhibit the lowest lacunarity values, reflecting their dense, uniformly ramified arbors with minimal large‑scale gaps. Primed microglia show slightly elevated λ̄, consistent with early structural irregularities associated with partial process withdrawal. Resolving microglia occupy an intermediate range, reflecting transitional morphologies during recovery. Activated and ameboid microglia display the highest lacunarity values, indicative of compact morphologies with large contiguous voids and reduced branching. These results highlight lacunarity as a sensitive multiscale descriptor of spatial heterogeneity, complementing fractal dimension by capturing differences in gap distribution and local irregularity across microglial states.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## **Module 5 — Texture Metrics via GLCM Analysis**

This module computes texture descriptors from microglial images using the Gray-Level Co-occurrence Matrix (GLCM) and integrates them into the broader morphometric analysis pipeline.

In addition to feature extraction, the module:

- computes GLCM-based metrics for each image
- organizes results into a structured summary table
- generates diagnostic and group-level visualizations
- produces a final multi-metric panel combining fractal dimension, lacunarity, and texture descriptors

The analysis is applied to preprocessed grayscale images and their corresponding binary masks, ensuring that texture is evaluated within the segmented microglial structure.

---

1. Conceptual Role in Microglial Analysis

GLCM-based texture metrics capture fine-scale intensity organization and local spatial relationships, complementing:

- fractal dimension (global complexity)
- lacunarity (spatial heterogeneity)

These features are particularly useful for detecting subtle morphological transitions that are not fully captured by global structural descriptors.

Typical trends observed in microglial datasets include:

- ramified morphologies → higher contrast, lower homogeneity
- compact morphologies → lower contrast, higher homogeneity
- intermediate states → transitional texture profiles

These associations depend on image quality and preprocessing and should be interpreted in context.

2. GLCM Feature Computation
'compute_glcm_features(gray_img, mask=None, ...)'

This function computes GLCM features from a grayscale image, optionally restricted to a segmented region of interest.

- Region of Interest (ROI)

For each image, the analysis is performed on a cropped region defined by the bounding box of the binary mask:

- reduces background influence
- focuses the analysis on the microglial structure
- improves robustness of texture estimation

A binary mask is applied to further suppress non-relevant pixels.

- Normalization

The grayscale image is:

- normalized to the range [0, 1]
- re-normalized within the masked region (if a mask is provided)

This ensures consistent intensity scaling across images and datasets.

- Quantization

The normalized image is discretized into a fixed number of gray levels by default: 32 levels

This step is required for GLCM computation and controls the resolution of texture analysis.

- GLCM computation

The gray-level co-occurrence matrix is computed using:

- multiple distances: [1, 2, 4]
- multiple angles: [0, π/4, π/2, 3π/4]

This multi-directional, multi-scale approach ensures:

- robustness to orientation
- reduced directional bias

- Feature extraction

The following features are computed and averaged across all distances and angles:

contrast (local intensity variation)
homogeneity (smoothness and uniformity)
energy (textural regularity)
correlation (linear dependency of pixel intensities)

- Edge-case handling

The function explicitly handles degenerate cases:

empty images or masks → all values set to NaN
constant-intensity regions:
- contrast = 0
- homogeneity = 1
- energy = 1
- correlation = 0

This ensures numerical stability and reproducibility.

3. GLCM Summary Table

For each image, the computed features are stored in a structured table:

- filename
- cell_state
- contrast
- homogeneity
- energy
- correlation

The table is:

- ordered according to the predefined biological state order (state_order)
- cleaned to remove auxiliary columns

and exported as:

'Module 5 - glcm_summary_table.csv'

4. Diagnostic Visualization

The module generates diagnostic plots showing each GLCM metric across all images:

- one subplot per metric
- values plotted against filenames
- useful for quick inspection of variability and outliers

Exported as:

'Module 5 - diagnostic_plots_for_glcm_metrics.png'

5. Group-Level Visualization

Each GLCM metric is further visualized using ordered plots:

- bar plots or boxplots depending on dataset size
- grouped by microglial activation state
- consistent color mapping across groups

These plots facilitate comparison of texture properties across biological conditions.

6. Integrated Morphometric Panel

The module concludes with a 2×2 summary panel combining key descriptors:

A — Fractal Dimension
B — Mean Lacunarity
C — GLCM Contrast
D — GLCM Homogeneity

This panel provides a compact, multidimensional view of microglial morphology, integrating:

- global complexity
- spatial heterogeneity
- fine-scale texture

Exported as:

'Module 5 - final_figure_panel_fd_lacunarity_glcm_metrics.png'

---

### **Role in the Pipeline**

This module extends the morphometric framework by incorporating intensity-based texture descriptors into the analysis.

Together with fractal dimension and lacunarity, GLCM features enable a richer and more comprehensive characterization of microglial morphology across activation states and experimental conditions.


In [ ]:
# ============================================================
# GLCM Feature Extraction
# ============================================================
# This function computes GLCM-based texture metrics for a
# grayscale microglial image. Features are averaged across
# multiple directions and distances to ensure rotational
# invariance and robustness across activation states.
#
# Design principles:
#   - Universal: works with any microglial activation state
#   - Reproducible: fixed distances and angles
#   - Minimalistic: only essential features are returned
#   - Robust: handles empty or low-contrast images safely
#
# Researchers may adjust distances or angles depending on
# resolution or biological scale of interest.
# ============================================================

print("\n[INFO] ===== Module 5 - GLCM - Based Texture Metrics =====")

def compute_glcm_features(
    gray_img,
    mask=None,
    distances=[1, 2, 4],
    angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
    levels=32
):
    """
    Compute GLCM features on a cropped grayscale ROI.
    Optionally uses a binary mask to suppress background influence.
    """
    gray = gray_img.astype(np.float32)

    if gray.size == 0:
        return {
            "contrast": np.nan,
            "homogeneity": np.nan,
            "energy": np.nan,
            "correlation": np.nan
        }

    # Normalize to [0, 1]
    gray = gray - gray.min()
    if gray.max() > 0:
        gray = gray / gray.max()

    # Apply mask if provided
    if mask is not None:
        mask = mask.astype(bool)
        if np.any(mask):
            vals = gray[mask]
            vmin, vmax = vals.min(), vals.max()

            if vmax > vmin:
                gray = (gray - vmin) / (vmax - vmin)
                gray = np.clip(gray, 0, 1)
            else:
                gray = np.zeros_like(gray)

            gray = gray * mask.astype(np.float32)
        else:
            return {
                "contrast": np.nan,
                "homogeneity": np.nan,
                "energy": np.nan,
                "correlation": np.nan
            }

    # Quantize
    gray_q = np.floor(gray * (levels - 1)).astype(np.uint8)

    if np.std(gray_q) == 0:
        return {
            "contrast": 0.0,
            "homogeneity": 1.0,
            "energy": 1.0,
            "correlation": 0.0
        }

    glcm = graycomatrix(
        gray_q,
        distances=distances,
        angles=angles,
        levels=levels,
        symmetric=True,
        normed=True
    )

    return {
        "contrast": float(np.mean(graycoprops(glcm, "contrast"))),
        "homogeneity": float(np.mean(graycoprops(glcm, "homogeneity"))),
        "energy": float(np.mean(graycoprops(glcm, "energy"))),
        "correlation": float(np.mean(graycoprops(glcm, "correlation")))
    }

# ============================================================
# GLCM Summary Table
# ============================================================
# This block computes GLCM texture metrics for all microglial images and stores them in a summary table.
#
# Columns:
#   filename | cell_state | contrast | homogeneity | energy | correlation
# ============================================================

glcm_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        img_resized = data["img_resized"]
        binary_uint8 = data["binary_uint8"]
        bbox = data["bbox"]

        r0, c0, r1, c1 = bbox
        gray = img_resized[r0:r1, c0:c1]
        binary_roi = binary_uint8[r0:r1, c0:c1].astype(bool)

        feats = compute_glcm_features(gray, mask=binary_roi)

        glcm_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            **feats
        })

    except Exception as e:
        glcm_rows.append({
            "filename": path_to_uid[path],
            "sample_id": path,
            "cell_state": cell_state,
            "contrast": None,
            "homogeneity": None,
            "energy": None,
            "correlation": None,
            "error": str(e)
        })

df_glcm = pd.DataFrame(glcm_rows)

# ------------------------------------------------------------
# Apply ordering
# ------------------------------------------------------------

df_glcm["order_key"] = df_glcm["cell_state"].apply(
    lambda x: state_order.index(x) if x in state_order else len(state_order)
)

df_glcm = df_glcm.sort_values(by=["order_key", "filename"]).drop(columns=["order_key"])

# ------------------------------------------------------------
# Final clean table
# ------------------------------------------------------------
df_glcm = df_glcm[[
    "filename",
    "sample_id",
    "cell_state",
    "contrast",
    "homogeneity",
    "energy",
    "correlation"
]]

# ------------------------------------------------------------
# EXPORT TABLE (GLCM Summary Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_glcm.to_csv(os.path.join(EXPORT_DIR, "Module 5 - glcm_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_glcm  

**Table 3. GLCM‑derived texture metrics across microglial activation states.**  
This table summarizes four second‑order texture features—contrast, homogeneity, energy, and correlation—computed from gray‑level co‑occurrence matrices (GLCMs) for each microglial sample. Resting microglia exhibit the highest contrast and energy values, reflecting the fine‑grained, highly ramified texture of their arborized processes. Primed and resolving microglia show intermediate texture signatures, consistent with partial process retraction and moderate structural irregularity. Activated and ameboid microglia display lower homogeneity and energy alongside elevated correlation, indicative of thicker, more uniform processes and reduced local intensity variation. These texture metrics capture activation‑dependent changes that are not fully reflected in geometric descriptors alone, highlighting the value of GLCM features as complementary indicators of microglial structural remodeling.


In [ ]:
# ============================================================
# Diagnostic Plots for GLCM Metrics
# ============================================================
# This block visualizes GLCM texture metrics across all microglial samples in the dataset.
#
# Each subplot shows:
#   - the metric values for each image
#   - filenames for identification
#
# These plots help reveal texture differences between
# microglial activation states:
#   - Resting microglia → higher contrast, lower homogeneity
#   - Activated / ameboid → high homogeneity, high energy
#   - Primed / resolving → intermediate profiles
#
# The goal is to provide a quick, interpretable overview of fine-scale intensity patterns across groups.
# ============================================================

metrics = ["contrast", "homogeneity", "energy", "correlation"]

df_diag = df_glcm.copy().replace({None: np.nan})

# Añadir sample_id único desde preprocessed_data
filename_to_sample = {}
for p in all_paths:
    d = preprocessed_data[p]
    filename_to_sample[(d["filename"], d["cell_state"])] = d["sample_id"]

df_diag["sample_id"] = [
    filename_to_sample.get((row["filename"], row["cell_state"]), row["filename"])
    for _, row in df_diag.iterrows()
]

# Orden consistente
df_diag["order_key"] = df_diag["cell_state"].apply(
    lambda x: state_order.index(x) if x in state_order else len(state_order)
)
df_diag = df_diag.sort_values(["order_key", "cell_state", "sample_id"]).reset_index(drop=True)
df_diag["xpos"] = np.arange(len(df_diag))

color_map = build_group_color_map(state_order)

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]

    for state in state_order:
        sub = df_diag[df_diag["cell_state"] == state].dropna(subset=[metric])
        if sub.empty:
            continue

        ax.plot(
            sub["xpos"],
            sub[metric],
            marker="o",
            linestyle="-",
            linewidth=1.4,
            markersize=4.5,
            color=color_map[state],
            label=state,
            alpha=0.9
        )

    ax.set_xticks(df_diag["xpos"])
    ax.set_xticklabels(df_diag["sample_id"], rotation=75, ha="right", fontsize=7)
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"{metric.capitalize()} across samples")
    ax.grid(axis="y", linestyle="--", alpha=0.3)

handles = [
    plt.Line2D([0], [0], color=color_map[state], marker="o", lw=2, label=state)
    for state in state_order if state in df_diag["cell_state"].unique()
]
fig.legend(handles=handles, title="Group", loc="upper right")

plt.tight_layout()
# ------------------------------------------------------------
# EXPORT FIGURE (Diagnostic Plots for GLCM Metrics)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 5 - diagnostic_plots_for_glcm_metrics.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()

**Figure 6. GLCM‑derived texture features across microglial activation states.**  
This multi‑panel figure shows four second‑order texture metrics—contrast, homogeneity, energy, and correlation—computed from gray‑level co‑occurrence matrices (GLCMs) for each microglial sample. Resting microglia exhibit the highest contrast and energy values, reflecting the fine‑grained, highly ramified texture characteristic of their complex arborization. Primed and resolving microglia display intermediate patterns, consistent with partial process retraction and moderate structural irregularity. Activated and ameboid microglia show lower homogeneity and energy alongside elevated correlation, indicative of thicker, more uniform processes and reduced local intensity variation. Together, these texture signatures reveal activation‑dependent shifts in microglial structural organization that complement geometric descriptors such as fractal dimension and lacunarity, providing an additional layer of sensitivity to subtle morphological remodeling.


In [ ]:
# ============================================================
# Ordered Bar Plots for GLCM Metrics
# ============================================================
# This version ensures full robustness by:
#   - replacing None values with NaN
#   - filtering invalid rows before plotting
#   - preserving microglial biological ordering
#   - keeping the visualization minimalistic and reproducible
#
# These plots help compare fine‑scale texture differences across:
#   - resting_microglia
#   - primed_microglia
#   - activated_microglia
#   - ameboid_microglia
#   - resolving_microglia
#
# GLCM metrics capture subtle intensity‑based signatures that complement fractal dimension and lacunarity.
# ============================================================

# Replace None with NaN for safe numeric plotting
df_plot = df_glcm.replace({None: np.nan})

# Metrics to visualize
metrics = ["contrast", "homogeneity", "energy", "correlation"]

# Deterministic color map per group
unique_states = df_plot["cell_state"].unique()
color_map = {
    state: plt.cm.tab10(i % 10)
    for i, state in enumerate(unique_states)
}

df_plot["color"] = df_plot["cell_state"].map(color_map)

metrics = ["contrast", "homogeneity", "energy", "correlation"]

for metric in metrics:
    plot_bars_or_boxplot(
        df=df_plot,
        value_col=metric,
        group_col="cell_state",
        filename_col="sample_id",
        state_order=state_order,
        ylabel=metric.capitalize(),
        xlabel="Image Filename / Group",
        title=f"{metric.capitalize()} Across Groups",
        save_path=os.path.join(EXPORT_DIR, f"Module 5 - ordered_bar_plot_glcm_{metric}.png")
    )

**Figure 7. GLCM texture metrics across microglial activation states: contrast, homogeneity, energy, and correlation.**  
This multi‑panel figure summarizes four second‑order texture descriptors derived from gray‑level co‑occurrence matrices (GLCMs), revealing complementary aspects of microglial structural organization across activation states.  
**Top panel – Contrast:** Resting microglia exhibit the highest contrast values, reflecting the fine‑grained, high‑frequency intensity variations produced by their dense and highly ramified processes. Contrast progressively decreases through primed and resolving states and reaches its lowest levels in activated and ameboid microglia, whose thick, compact processes generate smoother local intensity transitions.  
**Second panel – Homogeneity:** Homogeneity shows the inverse trend: resting microglia display the highest values, consistent with the smooth, continuous distribution of intensities generated by their arborized texture. Activated and ameboid microglia exhibit the lowest homogeneity, reflecting abrupt transitions and reduced structural regularity.  
**Third panel – Energy:** Energy, a measure of textural uniformity, is highest in resting microglia due to the repetitive fine‑scale patterns created by their extensive branching. Primed and resolving microglia show intermediate energy values, while activated and ameboid cells display the lowest energy, consistent with simplified, less repetitive intensity patterns.  
**Bottom panel – Correlation:** Correlation captures linear dependency between neighboring pixel intensities and is highest in ameboid and activated microglia, whose thick, homogeneous processes produce strong local intensity continuity. Resting microglia show the lowest correlation values, driven by the irregular, fine‑scale fluctuations characteristic of highly ramified arbors.  
Together, these four texture metrics provide a multidimensional view of microglial structural remodeling, capturing activation‑dependent changes that complement geometric descriptors such as fractal dimension and lacunarity.


In [ ]:
# ============================================================
# Final Figure Panel — FD, Lacunarity, and GLCM Metrics
# ============================================================
# This block generates a 2×2 panel summarizing the key morphometric descriptors for microglial morphology:
#
#   A — Fractal Dimension (ramification / complexity)
#   B — Mean Lacunarity (heterogeneity / gap distribution)
#   C — GLCM Contrast (fine‑scale intensity variation)
#   D — GLCM Homogeneity (texture smoothness / compactness)
#
# Design principles:
#   - Microglia‑specific: reflects activation‑dependent remodeling
#   - Reproducible: deterministic ordering and colors
#   - Minimalistic: only essential visual elements
#   - Publication‑ready: suitable for high‑impact journals
#
# This panel provides a compact, multidimensional summary of microglial structural organization across groups.
# ============================================================

def plot_on_axis_bars_or_boxplot(
    ax,
    df,
    value_col,
    group_col="cell_state",
    filename_col="sample_id",
    state_order=None,
    ylabel=None,
    title=None,
    image_threshold=5,
    group_threshold=5
):
    """
    Same logic as plot_bars_or_boxplot, but draws inside an existing axis.
    """
    df_plot = df.copy().dropna(subset=[value_col, group_col])

    if df_plot.empty:
        ax.set_title(title if title else value_col)
        ax.text(0.5, 0.5, "No valid data", ha="center", va="center")
        ax.axis("off")
        return

    # Group ordering
    if state_order is None:
        groups = list(df_plot[group_col].dropna().unique())
    else:
        groups = [g for g in state_order if g in df_plot[group_col].dropna().unique()]
        groups += [g for g in df_plot[group_col].dropna().unique() if g not in groups]

    df_plot[group_col] = pd.Categorical(df_plot[group_col], categories=groups, ordered=True)

    sort_cols = [group_col]
    if filename_col in df_plot.columns:
        sort_cols.append(filename_col)

    df_plot = df_plot.sort_values(sort_cols).reset_index(drop=True)

    color_map = build_group_color_map(groups)
    use_box = should_use_boxplot(
        df_plot,
        group_col=group_col,
        image_threshold=image_threshold,
        group_threshold=group_threshold
    )

    # --------------------------------------------------------
    # CASE 1 — bar plot per image
    # --------------------------------------------------------
    if not use_box:
        df_plot["color"] = df_plot[group_col].astype(object).map(color_map)

        ax.bar(
            x=np.arange(len(df_plot)),
            height=df_plot[value_col],
            color=df_plot["color"],
            edgecolor="black",
            linewidth=0.7
        )

        ax.set_xticks(np.arange(len(df_plot)))
        if filename_col in df_plot.columns:
            ax.set_xticklabels(df_plot[filename_col], rotation=75, ha="right", fontsize=7)

        ax.grid(axis="y", linestyle="--", alpha=0.3)

    # --------------------------------------------------------
    # CASE 2 — boxplot + individual points
    # --------------------------------------------------------
    else:
        light_palette = {g: lighten_color(color_map[g], 0.6) for g in groups}

        sns.boxplot(
            data=df_plot,
            x=group_col,
            y=value_col,
            hue=group_col,
            order=groups,
            palette=light_palette,
            dodge=False,
            width=0.55,
            showfliers=False,
            linewidth=1.2,
            ax=ax
        )

        sns.stripplot(
            data=df_plot,
            x=group_col,
            y=value_col,
            hue=group_col,
            order=groups,
            palette=color_map,
            dodge=False,
            jitter=0.12,
            alpha=0.9,
            size=4.5,
            edgecolor="black",
            linewidth=0.4,
            ax=ax
        )

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

        ax.tick_params(axis="x", rotation=45)
        ax.grid(axis="y", linestyle="--", alpha=0.3)

    ax.set_title(title if title else value_col, fontsize=12)
    ax.set_ylabel(ylabel if ylabel else value_col)

# ------------------------------------------------------------
# Prepare ordered dataframes
# ------------------------------------------------------------
df_fd_plot = df_fd.copy().dropna(subset=["fractal_dimension"])
df_lac_plot = df_lac.copy().dropna(subset=["lacunarity_mean"])
df_glcm_plot = df_glcm.copy().dropna(subset=["contrast", "homogeneity"])

# ------------------------------------------------------------
# Create the 2×2 panel
# ------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

# ------------------------------------------------------------
# Panel A — Fractal Dimension
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[0],
    df=df_fd_plot,
    value_col="fractal_dimension",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="FD",
    title="A — Fractal Dimension"
)

# ------------------------------------------------------------
# Panel B — Mean Lacunarity
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[1],
    df=df_lac_plot,
    value_col="lacunarity_mean",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Λ",
    title="B — Mean Lacunarity"
)

# ------------------------------------------------------------
# Panel C — GLCM Contrast
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[2],
    df=df_glcm_plot,
    value_col="contrast",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Contrast",
    title="C — GLCM Contrast"
)

# ------------------------------------------------------------
# Panel D — GLCM Homogeneity
# ------------------------------------------------------------
plot_on_axis_bars_or_boxplot(
    ax=axes[3],
    df=df_glcm_plot,
    value_col="homogeneity",
    group_col="cell_state",
    filename_col="sample_id",
    state_order=state_order,
    ylabel="Homogeneity",
    title="D — GLCM Homogeneity"
)

# ------------------------------------------------------------
# Legend (shared across all panels)
# ------------------------------------------------------------
present_states = []
for df_tmp in [df_fd_plot, df_lac_plot, df_glcm_plot]:
    present_states.extend(df_tmp["cell_state"].dropna().unique().tolist())

unique_states = []
for s in state_order:
    if s in present_states and s not in unique_states:
        unique_states.append(s)

color_map = {state: plt.cm.tab10(i % 10) for i, state in enumerate(unique_states)}

handles = [
    plt.Line2D([0], [0], color=color_map[state], lw=8, label=state)
    for state in unique_states
]
fig.legend(handles=handles, title="Group", loc="upper right")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE
# ------------------------------------------------------------
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 5 - final_figure_panel_fd_lacunarity_glcm_metrics.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\n[INFO] ===== Module 5 Completed =====")

**Figure 8. Multidomain morphometric descriptors reveal consistent activation‑dependent gradients in microglial structure.**  
This four‑panel figure integrates geometric, fractal, and texture‑based metrics to characterize microglial morphology across activation states.  
**(A) Fractal Dimension (FD):** Resting microglia exhibit the highest FD values, reflecting their dense, highly ramified arbors. FD progressively decreases through primed and resolving states and reaches its lowest levels in activated and ameboid microglia, whose compact morphologies lack fine‑scale branching.  
**(B) Mean Lacunarity (Λ):** Lacunarity shows the inverse trend: ameboid and activated microglia display the highest Λ values, indicative of large structural gaps and reduced arborization. Resting microglia show minimal lacunarity, consistent with their uniform, space‑filling branching patterns.  
**(C) GLCM Contrast:** Resting microglia exhibit the highest contrast values, driven by the fine‑grained intensity variations produced by their complex arborization. Activated and ameboid microglia show markedly lower contrast, reflecting smoother, more homogeneous textures associated with simplified morphologies.  
**(D) GLCM Homogeneity:** Homogeneity mirrors the structural regularity of each state: resting microglia show the highest values, while ameboid microglia exhibit the lowest, consistent with abrupt intensity transitions and reduced structural continuity.  
Together, these complementary metrics reveal a coherent multiscale signature of microglial activation, demonstrating that geometric complexity, spatial heterogeneity, and texture organization shift in parallel as microglia transition from resting to activated and ameboid states.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 6 — Multiscale Fractal Spectrum (MFS)

**Operational definition.** In this workflow, MFS denotes the scale-dependent profile of local box-counting slopes:

$$\mathrm{MFS}(s)=-\frac{d\log N(s)}{d\log s},$$

where N(s) is the number of boxes containing foreground at box size s. Here **spectrum means variation across spatial scales**. It is a name defined for this descriptor, not a claim that the algorithm estimates a conventional multifractal singularity spectrum f(alpha) or generalized dimensions D(q).

The former expression log(N(s))/log(1/s) has been removed: it was a ratio rather than a local slope. The corrected values require reprocessing and are not interchangeable with the earlier results.

### Computation

Input is the FD-oriented skeleton, skeleton_fd_uint8. The local_boxcount_slopes function:

1. Selects distinct valid sizes from 2, 4, 8, 16, 32, 64, 128 and 256 pixels.
2. Anchors boxes at the top-left and zero-pads incomplete right/bottom boxes so border foreground is retained.
3. Counts boxes with at least one foreground pixel.
4. Computes minus numpy.gradient(log(counts), log(scales)): centered finite differences at interior scales and one-sided differences at the two endpoints.

An empty image or fewer than two valid scales produces undefined slopes (NaN), not an invented dimension. There is no smoothing, q-order partition function or Legendre transform in this implementation. Finite-image slopes depend on the grid origin, resolution, selected sizes and segmentation; they are descriptive estimates, not proof of fractality over an asymptotic scaling range.

### Outputs and terminology

The public descriptor name **Multiscale Fractal Spectrum (MFS)** is retained. Internal columns local_fd_scale_2, local_fd_scale_4, etc. identify the corrected calculation and distinguish it from the former mfs_scale_* log-ratio columns. The internal prefix does not designate a second descriptor.

- Module 6 - local_fd_table.csv: one row per image, with identifiers, group and local slopes.
- Module 6 - local_fd_curves_per_image.png: individual MFS profiles.
- Module 6 - comparative_local_fd_by_group.png: mean profile per group; this figure does not show uncertainty intervals.
- Module 6 - heatmap_of_the_local_fd.png: samples by sampled scales.

The profile complements the single global FD value and is included in the master matrix and exploratory model. Module 9 excludes local_fd_scale_* from the standard scalar tests; no separate inferential test of the MFS profile is implemented. Biological conclusions require experimental data and appropriate statistical evaluation; broad or flat profiles do not by themselves establish an activation state.


In [ ]:
# ============================================================
#  Multiscale Fractal Spectrum (MFS)
# ============================================================
# MFS is the operational name for the local box-counting slope profile.
# MFS(s) = -d log N(s) / d log s, not a conventional multifractal spectrum.
# Numerical slopes use centered differences internally and one-sided endpoints.

print("\n[INFO] ===== Module 6 - Multiscale Fractal Spectrum (MFS) =====")

def local_boxcount_slopes(binary_img, scales=(2, 4, 8, 16, 32, 64, 128, 256)):
    """Return -gradient(log N, log s) at valid box sizes in pixels.

    Boxes are anchored at the top-left and zero-padded at right/bottom, so
    foreground at image borders is retained. Interior derivatives use NumPy's
    centered finite differences; first/last use one-sided differences. Empty
    images or fewer than two valid scales produce NaNs, never invented slopes.
    This scale profile is not a multifractal f(alpha) or D(q) spectrum.
    """
    binary = np.asarray(binary_img, dtype=bool)
    if binary.ndim != 2:
        raise ValueError("Box counting requires a 2-D image.")
    h, w = binary.shape
    valid = sorted({int(s) for s in scales if int(s) == s and 1 < s <= min(h, w)})
    counts = []
    for s in valid:
        padded = np.pad(binary, ((0, (-h) % s), (0, (-w) % s)))
        boxes = padded.reshape(padded.shape[0] // s, s, padded.shape[1] // s, s)
        counts.append(np.count_nonzero(boxes.any(axis=(1, 3))))
    slopes = np.full(len(valid), np.nan)
    if len(valid) >= 2 and all(c > 0 for c in counts):
        slopes = -np.gradient(np.log(np.asarray(counts, float)), np.log(valid))
    return slopes, valid

# ============================================================
# Multiscale Fractal Spectrum (MFS) Table
# ============================================================
# Input: FD-oriented skeleton. Empty or insufficient-scale inputs yield NaN.
# Internal local_fd_scale_* columns distinguish this corrected implementation
# from the former log-ratio columns mfs_scale_*.

local_fd_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        skel = data["skeleton_fd_uint8"].astype(bool)
        spectrum, scales = local_boxcount_slopes(skel)

        local_fd_rows.append({
            "filename": data["filename"],
            "cell_state": data["cell_state"],
            **{f"local_fd_scale_{s}": spectrum[j] for j, s in enumerate(scales)}
        })

    except Exception as e:
        print("Error processing:", path, e)
        continue

df_local_fd = pd.DataFrame(local_fd_rows)

# ------------------------------------------------------------
# EXPORT TABLE (MFS Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_local_fd.to_csv(os.path.join(EXPORT_DIR, "Module 6 - local_fd_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_local_fd 

**Table 4. Multiscale Fractal Spectrum (MFS): corrected local box-counting slopes.**

Rows identify images and groups. Columns local_fd_scale_* contain finite-difference estimates of −d log N(s)/d log s at the indicated box sizes in preprocessed pixels. This operational multiscale spectrum is not a conventional multifractal spectrum. Missing values identify empty or insufficient-scale inputs. Numerical values from the previous log-ratio implementation must be recomputed.


In [ ]:
# ============================================================
# MFS Curves per Image
# ============================================================
# This block visualizes the Multiscale Fractal Spectrum for each microglial image.
#
# Each subplot shows:
#   - the scales used (box sizes)
#   - the corresponding multiscale fractal descriptors
#   - the filename for identification
#
# Biological interpretation:
#   - Resting microglia → broad, high‑variance spectra
#   - Primed microglia → reduced fine‑scale complexity
#   - Activated microglia → flatter spectra (compact morphology)
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re‑emergence of multiscale structure
#
# These curves provide a scale-resolved view of microglial complexity that complements FD, lacunarity, and GLCM metrics.
# ============================================================

n = len(df_local_fd)
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, row in df_local_fd.iterrows():
    ax = axes[i]

    scales = [int(col.split("_")[-1]) for col in df_local_fd.columns if col.startswith("local_fd_scale_")]
    values = [row[f"local_fd_scale_{s}"] for s in scales]

    ax.plot(scales, values, marker="o")
    ax.set_title(row["filename"], fontsize=9)
    ax.set_xlabel("Scale (box size)")
    ax.set_ylabel("MFS value")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, "Module 6 - local_fd_curves_per_image.png"), dpi=300, bbox_inches="tight")
plt.show()

**Figure 9. Individual MFS profiles across spatial scales.**

Each subplot shows one image's corrected local box-counting slopes against box size. The end values use one-sided differences; interior values use centered differences. Curves are descriptive signatures of the analyzed skeletons. Their shape depends on resolution, segmentation, grid origin and scale selection; it is not a diagnostic label for microglial function.


In [ ]:
# ============================================================
# Comparative Multiscale Fractal Spectrum by Group
# ============================================================
# This block compares the Multiscale Fractal Spectrum across groups.
#
# Interpretation:
#   - Resting microglia → broad, high‑variance spectra
#   - Primed microglia → reduced fine‑scale complexity
#   - Activated microglia → flatter spectra (compact morphology)
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re‑emergence of multiscale structure
#
# This visualization highlights how microglial complexity
# evolves across spatial scales and differs between states.
# ============================================================

# Melt table into long format
df_long = df_local_fd.melt(
    id_vars=["filename", "cell_state"],
    var_name="scale_label",
    value_name="MFS_s"
)

# Extract numeric scale
df_long["scale"] = df_long["scale_label"].str.extract(r"(\d+)").astype(int)

# Compute mean per cell_state × scale
df_grouped = df_long.groupby(["cell_state", "scale"], as_index=False)["MFS_s"].mean()

# Deterministic colors per mgroup
unique_states = df_grouped["cell_state"].unique()
color_map = {state: plt.cm.tab10(i % 10) for i, state in enumerate(unique_states)}

plt.figure(figsize=(8, 6))

for state in unique_states:
    sub = df_grouped[df_grouped["cell_state"] == state]
    plt.plot(sub["scale"], sub["MFS_s"], marker="o", color=color_map[state], label=state)

plt.xlabel("Scale (box size)")
plt.ylabel("Mean MFS value")
plt.title("Multiscale Fractal Spectrum by Group")
plt.legend(title="Group")
plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Comparative MFS by group)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 6 - comparative_local_fd_by_group.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()

**Figure 10. Group mean MFS profiles.**

At each sampled scale, the plotted value is the mean available local box-counting slope within the user-defined group. Lines summarize scale-dependent morphology; no confidence interval or repeated-measures test of these MFS profiles is shown. Evaluate between-image variability and sample size before attributing differences to an experimental condition.


In [ ]:
# ============================================================
# Heatmap of the Multiscale Fractal Spectrum
# ============================================================
# This heatmap provides a compact visualization of the MFS across all microglial images. 
# Each row corresponds to one image, and each column corresponds to a multiscale descriptor (FD_s at a given box size).
#
# Biological interpretation:
#   - Resting microglia → high values at fine scales, broad profile
#   - Primed microglia → reduced fine-scale complexity
#   - Activated microglia → flatter, lower MFS across scales
#   - Ameboid microglia → minimal multiscale variation
#   - Resolving microglia → re-expansion of multiscale structure
#
# Design principles:
#   - Microglia-specific but universally applicable
#   - Reproducible: deterministic ordering and color mapping
#   - Minimalistic: only essential information is shown
#   - Customizable: colormap, ordering, normalization
# ============================================================

# ------------------------------------------------------------
# Select only numeric MFS columns
# (all columns starting with "local_fd_scale_")
# ------------------------------------------------------------
local_fd_cols = [col for col in df_local_fd.columns if col.startswith("local_fd_scale_")]
numeric_data = df_local_fd[local_fd_cols].values

# ------------------------------------------------------------
# Create the heatmap
# ------------------------------------------------------------
plt.figure(figsize=(10, 6))

plt.imshow(numeric_data, cmap="viridis", aspect="auto")
plt.colorbar(label="MFS value")

# Row labels: filenames
plt.yticks(
    ticks=np.arange(len(df_local_fd)),
    labels=df_local_fd["filename"],
    fontsize=7
)

# Column labels: scale descriptors
plt.xticks(
    ticks=np.arange(len(local_fd_cols)),
    labels=local_fd_cols,
    rotation=45,
    ha="right",
    fontsize=8
)

plt.title("MFS — Heatmap")
plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Heatmap of the MFS)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 6 - heatmap_of_the_local_fd.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()  

print("\n[INFO] ===== Module 6 Completed =====")

**Figure 11. MFS heatmap of corrected local slopes.**

Rows are analyzed images and columns are sampled box sizes. Colour represents the local box-counting slope at that scale. The heatmap visualizes variation within and between user-defined groups; it does not establish a hierarchy of activation states or measure multifractal singularities. See Module 6 for the finite-difference definition and boundary handling.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 7 — Graph-Based Morphometrics from Skeletonized Structures

The input is skeleton_visual_uint8. Each foreground pixel becomes one graph node; undirected edges connect all neighboring foreground pixels under deterministic 8-connectivity, with each edge created once. Counts therefore describe the **pixel graph**, not an anatomically reduced tree of branches.

### Descriptor definitions

| Column | Meaning |
|---|---|
| endpoints | Foreground pixels with exactly one foreground neighbor. |
| junction_pixel_count | Foreground pixels with at least three neighbors. A single anatomical junction may span several such pixels. |
| total_nodes | Number of foreground pixel nodes. |
| total_edges | Number of undirected neighbor connections, counted once. This is not physical or Euclidean length. |
| diameter | Approximate unweighted diameter of the largest component, obtained using two breadth-first traversals. |
| avg_degree | Mean node degree over the complete graph. |
| avg_junction_degree | Mean degree restricted to nodes with degree at least three; zero when no such nodes exist. |
| endpoint_density | endpoints / total_nodes. |
| junction_pixel_density | junction_pixel_count / total_nodes. |
| junction_to_endpoint_ratio | junction_pixel_count / endpoints; undefined if endpoints is zero. |
| largest_component_nodes | Node count of the largest connected component. |
| n_components | Number of connected components. |

**Changes in interpretation.** bifurcations has been renamed junction_pixel_count. The density and ratio have been renamed consistently. This does not introduce branch-event clustering or anatomical bifurcation counting. total_length has been removed because it previously duplicated total_edges exactly. No geometric length descriptor has been added.

The diameter is an approximation, especially for graphs with cycles; it is not an exact all-pairs diameter or a calibrated physical distance. The former explanation about returning NaN above 2000 nodes no longer applies. Empty skeletons retain zero counts and undefined applicable ratios/diameter.

### Outputs

The module exports the per-image graph table, a pixel-node overlay, the integrated df_master_basic table, nine comparative descriptor panels, and group-mean profiles for endpoints, junction_pixel_count, total_nodes, total_edges, diameter and avg_degree. Tables merge on the unique relative-path filename and cell_state. All groups are labels from the input organization.

These descriptors summarize connectivity and pixel-scale organization. Their relationship to a biological state must be evaluated rather than assumed from the feature name or the synthetic example.


In [ ]:
# ============================================================
# Graph-Based Morphometrics
# ============================================================
# This module extracts graph-theoretic descriptors from the skeleton produced in the preprocessing pipeline.
#
# Pixel-level descriptors; junction pixels are not anatomical branch events.
# Full undirected 8-connectivity; total_edges is an edge count, not length.

print("\n[INFO] ===== Module 7 - Graph-Based Morphometrics =====")

# ------------------------------------------------------------
# 8-connectivity kernel for neighbor counting
# ------------------------------------------------------------
def neighbor_count(skel):
    kernel = np.ones((3, 3), dtype=np.int16)
    kernel[1, 1] = 0
    conv = convolve(skel.astype(np.uint8), kernel, mode="constant", cval=0)
    return conv



def skeleton_to_graph(skel):
    G = nx.Graph()
    skel = skel.astype(bool)

    ys, xs = np.where(skel)
    nodes = list(zip(ys, xs))
    G.add_nodes_from(nodes)

    node_set = set(nodes)

    # Only 4 duplicates "to the front" to avoid duplicate edges
    forward_neighbors = [
        (-1,  1),
        ( 0,  1),
        ( 1,  1),
        ( 1,  0),
    ]

    for y, x in nodes:
        for dy, dx in forward_neighbors:
            nb = (y + dy, x + dx)
            if nb in node_set:
                G.add_edge((y, x), nb)

    return G



def fast_largest_component_diameter(G):
    """
    Fast approximate graph diameter using two BFS passes.

    For skeleton-like graphs this is much faster than nx.diameter().
    It avoids returning NaN when the largest component has >2000 nodes.
    """

    if G.number_of_nodes() == 0:
        return np.nan

    try:
        largest_cc = max(nx.connected_components(G), key=len)
        subG = G.subgraph(largest_cc).copy()

        if subG.number_of_nodes() == 0:
            return np.nan

        if subG.number_of_nodes() == 1:
            return 0

        # First BFS: arbitrary node -> farthest node
        start = next(iter(subG.nodes))
        lengths = nx.single_source_shortest_path_length(subG, start)
        farthest = max(lengths, key=lengths.get)

        # Second BFS: farthest node -> farthest distance
        lengths = nx.single_source_shortest_path_length(subG, farthest)
        diameter_approx = max(lengths.values())

        return int(diameter_approx)

    except Exception:
        return np.nan



def graph_morphometrics(skel):
    skel = skel.astype(bool)

    if not np.any(skel):
        return dict(
            endpoints=0,
            junction_pixel_count=0,
            total_nodes=0,
            total_edges=0,
            diameter=np.nan,
            avg_degree=np.nan,
            avg_junction_degree=np.nan,
            endpoint_density=np.nan,
            junction_pixel_density=np.nan,
            junction_to_endpoint_ratio=np.nan,
            largest_component_nodes=0,
            n_components=0
        )

    neigh = neighbor_count(skel)

    endpoints = int(np.sum(skel & (neigh == 1)))
    junction_pixel_count = int(np.sum(skel & (neigh >= 3)))

    G = skeleton_to_graph(skel)

    total_nodes = G.number_of_nodes()
    total_edges = G.number_of_edges()

    # Connected components
    try:
        components = list(nx.connected_components(G))
        n_components = len(components)
        largest_component_nodes = len(max(components, key=len)) if components else 0
    except Exception:
        n_components = np.nan
        largest_component_nodes = np.nan

    # Fast diameter instead of nx.diameter cutoff at 2000 nodes
    diameter = fast_largest_component_diameter(G)

    # Degree metrics
    if total_nodes > 0:
        degrees = np.array([deg for _, deg in G.degree()], dtype=float)

        # This will usually be close to 2 in skeleton graphs.
        avg_degree = float(np.mean(degrees))

        # More biologically useful: average degree only at branching nodes.
        junction_degrees = degrees[degrees >= 3]
        if len(junction_degrees) > 0:
            avg_junction_degree = float(np.mean(junction_degrees))
        else:
            avg_junction_degree = 0.0

        endpoint_density = endpoints / total_nodes
        junction_pixel_density = junction_pixel_count / total_nodes

        if endpoints > 0:
            junction_to_endpoint_ratio = junction_pixel_count / endpoints
        else:
            junction_to_endpoint_ratio = np.nan

    else:
        avg_degree = np.nan
        avg_junction_degree = np.nan
        endpoint_density = np.nan
        junction_pixel_density = np.nan
        junction_to_endpoint_ratio = np.nan

    return dict(
        endpoints=endpoints,
        junction_pixel_count=junction_pixel_count,
        total_nodes=total_nodes,
        total_edges=total_edges,
        diameter=diameter,
        avg_degree=avg_degree,
        avg_junction_degree=avg_junction_degree,
        endpoint_density=endpoint_density,
        junction_pixel_density=junction_pixel_density,
        junction_to_endpoint_ratio=junction_to_endpoint_ratio,
        largest_component_nodes=largest_component_nodes,
        n_components=n_components
    )


# ============================================================
# Apply graph metrics to all group images
# ============================================================

graph_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)

        metrics = graph_morphometrics(skel)

        graph_rows.append({
            "filename": data["filename"],
            "cell_state": data["cell_state"],
            **metrics
        })

    except Exception as e:
        print("Error processing:", path, e)
        continue

df_graph = pd.DataFrame(graph_rows)

# ------------------------------------------------------------
# EXPORT TABLE (Graph-Based Morphometrics)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_graph.to_csv(os.path.join(EXPORT_DIR, "Module 7 - graph_based_morphometrics_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_graph

**Table 5. Pixel-graph descriptors of the analyzed skeletons.**

junction_pixel_count counts pixels with at least three 8-neighbors, not distinct anatomical branch events. total_edges is an undirected edge count; the duplicate total_length is absent. Density and junction-to-endpoint ratio columns follow the definitions in Module 7. diameter is an approximate unweighted extent of the largest component. Identifiers are relative paths, preserving distinct images with identical basenames.


In [ ]:
# ============================================================
# Graph Overlay on Skeletonized Microglial Structures
# ============================================================
# This block visualizes the graph representation of each microglial skeleton by overlaying graph nodes on top of the skeleton.
#
# Biological interpretation:
#   - Resting microglia → dense arbor, many nodes, rich topology
#   - Primed microglia → reduced distal branching
#   - Activated microglia → compact, simplified graph
#   - Ameboid microglia → minimal or near-zero graph structure
#   - Resolving microglia → re-expansion of graph connectivity
#
# Design principles:
#   - Microglia-specific but universally applicable
#   - Reproducible: deterministic skeleton + graph extraction
#   - Minimalistic: nodes only (edges optional for clarity)
# ============================================================

# Number of images
n = len(all_paths)

# Dynamic grid: 4 columns, as many rows as needed
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, (path, cell_state) in enumerate(zip(all_paths, all_cell_states)):

    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)
        G = skeleton_to_graph(skel)

        ax = axes[i]
        ax.imshow(skel, cmap="gray")
        ax.set_title(f"{data['filename']} ({data['cell_state']})", fontsize=9)
        ax.axis("off")

        ys = [node[0] for node in G.nodes()]
        xs = [node[1] for node in G.nodes()]

        ax.scatter(xs, ys, s=1, c="red")

    except Exception as e:
        print("Error processing:", path, e)
        continue

# Hide unused subplots (if any)
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Graph Overlay on Skeletonized Microglial Structures)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 7 - graph_overlay_on_skeletonized_microglial_structures.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show()  

**Figure 12. Pixel-graph nodes overlaid on the visual skeleton.**

The grayscale skeleton is overlaid with foreground pixel nodes in red. Graph adjacency follows 8-connectivity. Markers are pixel nodes, not automatically identified anatomical branching events. Inspect disconnected processes, segmentation artefacts and neighboring structures before interpreting the associated graph table.


In [ ]:
# ============================================================
# Build Master DataFrame (df_master_basic)
# ============================================================
# This merges all morphometric tables into a single unified
# dataframe indexed by filename + cell_state.
# ============================================================

# Start with fractal dimension table
df_master_basic = df_fd.copy()

# Merge lacunarity
df_master_basic = df_master_basic.merge(df_lac, on=["filename", "cell_state"], how="left")

# Merge GLCM metrics
df_master_basic = df_master_basic.merge(df_glcm, on=["filename", "cell_state"], how="left")

# Merge Multiscale Fractal Spectrum
df_master_basic = df_master_basic.merge(df_local_fd, on=["filename", "cell_state"], how="left")

# Merge graph-based morphometrics
df_master_basic = df_master_basic.merge(df_graph, on=["filename", "cell_state"], how="left")

# ------------------------------------------------------------
# EXPORT TABLE (Build Master DataFrame — df_master_basic)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_master_basic.to_csv(os.path.join(EXPORT_DIR, "Module 7 - build_master_dataframe_df_master_basic.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_master_basic 

**Table 6. Integrated feature matrix before Sholl integration.**

The table joins global FD, lacunarity, GLCM, corrected MFS slopes and pixel-graph descriptors by unique relative-path filename and group. It contains junction_pixel_count and total_edges, with no duplicate total_length. MFS uses local_fd_scale_* internally. The joined data feed later descriptive plots and are incorporated into the universal master dataframe.


In [ ]:
# Use the master dataframe for graph-based barplots
df_graph_diag = df_master_basic.copy()

# Ensure a valid cell_id exists
if "cell_id" not in df_graph_diag.columns:
    if "filename" in df_graph_diag.columns:
        df_graph_diag["cell_id"] = df_graph_diag["filename"]
    else:
        df_graph_diag["cell_id"] = df_graph_diag.index.astype(str)

# ============================================================
# Comparative Plots of Graph Morphometrics
# ============================================================

metrics = [
    "endpoints",
    "junction_pixel_count",
    "endpoint_density",
    "junction_pixel_density",
    "junction_to_endpoint_ratio",
    "avg_junction_degree",
    "diameter",
    "total_nodes",
    "total_edges"
]

# Ensure consistent naming
if "stage" in df_graph_diag.columns and "cell_state" not in df_graph_diag.columns:
    df_graph_diag["cell_state"] = df_graph_diag["stage"]

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    plot_on_axis_bars_or_boxplot(
        ax=axes[i],
        df=df_graph_diag,
        value_col=metric,
        group_col="cell_state",
        filename_col="cell_id",
        state_order=state_order,
        ylabel="Value",
        title=metric.replace("_", " ").capitalize()
    )

# Hide unused subplots
for j in range(len(metrics), len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Comparative Plots of Graph Morphometrics)
# ------------------------------------------------------------
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 7 - comparative_plots_of_graph_morphometrics_microglia.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

**Figure 13. Comparative pixel-graph descriptors.**

The nine panels show endpoints, junction_pixel_count, endpoint_density, junction_pixel_density, junction_to_endpoint_ratio, avg_junction_degree, approximate diameter, total_nodes and total_edges. The plotting utility selects per-image bars or boxplots with individual observations according to dataset structure. These descriptive comparisons do not by themselves establish statistical significance or anatomical branch counts.


In [ ]:
# Use the master dataframe for group-level graph morphometrics
df_graph_diag = df_master_basic.copy()

# ============================================================
# Group-Level Comparison of Graph Morphometrics
# Universal version: works with individual images or groups
# ============================================================

metrics = [
    "endpoints",
    "junction_pixel_count",
    "total_nodes",
    "total_edges",
    "diameter",
    "avg_degree"
]

# Ensure consistent naming
if "stage" in df_graph_diag.columns and "cell_state" not in df_graph_diag.columns:
    df_graph_diag["cell_state"] = df_graph_diag["stage"]

# Keep only metrics that exist in the dataframe
metrics_existing = [m for m in metrics if m in df_graph_diag.columns]

# Remove metrics that are completely empty
metrics_existing = [
    m for m in metrics_existing
    if not df_graph_diag[m].dropna().empty
]

# Identify groups in the same order as the dataset loader
groups = [g for g in state_order if g in df_graph_diag["cell_state"].unique()]
groups += [g for g in df_graph_diag["cell_state"].unique() if g not in groups]

# Deterministic color map per group
color_map = {state: plt.cm.tab10(i % 10) for i, state in enumerate(groups)}

plt.figure(figsize=(12, 6))

for state in groups:
    subset = df_graph_diag[df_graph_diag["cell_state"] == state]

    if subset.empty:
        continue

    mean_vals = subset[metrics_existing].mean(numeric_only=True)

    plt.plot(
        metrics_existing,
        mean_vals[metrics_existing],
        marker="o",
        label=state,
        color=color_map[state]
    )

# Decide automatically whether log scale is useful
all_values = df_graph_diag[metrics_existing].values.flatten()
all_values = all_values[np.isfinite(all_values)]
all_values = all_values[all_values > 0]

if len(all_values) > 0:
    value_ratio = all_values.max() / all_values.min()

    if value_ratio > 100:
        plt.yscale("log")
        ylabel = "Mean value (log scale)"
    else:
        ylabel = "Mean value"
else:
    ylabel = "Mean value"

plt.xticks(rotation=45, ha="right")
plt.ylabel(ylabel)
plt.title("Graph Morphometrics — Group-Level Comparison")
plt.legend(title="Group", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()

plt.savefig(
    os.path.join(EXPORT_DIR, "Module 7 - group_level_comparison_of_graph_morphometrics.png"),
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("\n[INFO] ===== Module 7 Completed =====")

**Figure 14. Group mean profiles of six pixel-graph descriptors.**

The figure compares endpoints, junction_pixel_count, total_nodes, total_edges, approximate diameter and avg_degree. A logarithmic axis may be selected automatically when positive values span a wide range. Features have different meanings and units; joining their means with lines is a visual summary, not a common physical measurement. total_length is excluded because it duplicated total_edges.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 8 — Sholl Analysis

Sholl profiles describe how **geometric crossings of a skeleton with concentric circles** vary with radius. This module uses skeleton_visual_uint8, the same representation as the graph module.

### Centre and radii

The automatic centre is the arithmetic centroid of foreground skeleton pixels. It is reproducible for a fixed skeleton, but is not a detected soma centre and may be influenced by asymmetric or disconnected processes. The scientific function accepts an explicit centre (x, y) if users choose to supply one; this notebook's batch call uses the centroid.

The default radial step is **10 preprocessed pixels**, not 5. Radii begin at that positive step. With max_radius=None, the final ring lies strictly outside the most distant foreground pixel; it is not set to the image diagonal. Coordinates refer to the current 1024 × 1024 preprocessed grid, without automatic micrometre calibration.

### Counting intersections

Each connection between neighboring skeleton pixels is treated as a straight segment. Circle–segment intersections are solved analytically, including crossings between pixel centres. The algorithm can retain two crossings on one chord.

For Sholl only, a diagonal connection is suppressed when an orthogonal connecting pixel is present, avoiding redundant corner shortcuts. The full 8-neighbor graph remains unchanged for Module 7 descriptors.

Counting conventions are explicit:

- Tangencies are excluded.
- A shared pixel exactly on a ring counts one location when incident segments reach both sides.
- A terminal pixel exactly on a ring counts once when its incident segment lies inside.
- An isolated pixel has no crossing.

This replaces the former count of pixels satisfying abs(distance − radius) < 0.5. The overlay uses the **same geometric crossing coordinates**, rather than showing markers from the former annulus approximation.

### Summary quantities and tables

- max_intersections: largest sampled crossing count.
- radius_of_max: first sampled radius attaining that maximum; undefined when every count is zero.
- auc: trapezoidal integral over the sampled positive-radius interval; no artificial observation at radius zero is added.
- decay_rate: ordinary linear-regression slope of crossing counts versus radius; not an exponential decay constant.

The full table preserves radii and counts, the summary table contains scalar descriptors, and the wide table provides one sholl_<radius> column per batch-wide sampled radius. Since automatic profiles include a ring outside the whole skeleton, subsequent radii can validly be padded with zero. This rule assumes complete morphology within the analyzed image; it cannot recover processes cut by the field of view.

The universal master dataframe merges Sholl summary and wide tables with the other feature tables, preserving unique cell_id values. Empty or degenerate inputs yield documented empty/undefined outputs. The algorithm has synthetic geometric tests; experimental centre placement, segmentation and field-of-view truncation still require visual assessment.


In [ ]:
# ============================================================
# Skeleton Centroid
# ============================================================
# Computes the centroid of a skeletonized morphology.
# Used as the universal anchor for Sholl analysis.
# ============================================================

print("\n[INFO] ===== Module 8 - Batch Sholl Analysis =====")

def skeleton_centroid(skel):
    skel = skel.astype(bool)

    ys, xs = np.where(skel)

    if len(xs) == 0:
        return None, None

    cx = xs.mean()
    cy = ys.mean()

    return cx, cy


# ============================================================
# Sholl Analysis
# ============================================================

def _sholl_geometry(skel):
    """Pixel graph for Sholl, suppressing diagonal shortcuts around corners.

    The full 8-neighbor graph is retained for graph descriptors. For Sholl,
    a diagonal is redundant when either orthogonal connecting pixel exists.
    """
    graph = skeleton_to_graph(np.asarray(skel, dtype=bool))
    for u, v in list(graph.edges):
        if u[0] != v[0] and u[1] != v[1]:
            if (u[0], v[1]) in graph or (v[0], u[1]) in graph:
                graph.remove_edge(u, v)
    nodes = list(graph.nodes)
    xy = np.asarray([(x, y) for y, x in nodes], float).reshape(-1, 2)
    index = {node: i for i, node in enumerate(nodes)}
    edges = np.asarray([(index[u], index[v]) for u, v in graph.edges], int).reshape(-1, 2)
    return graph, nodes, xy, edges

def sholl_intersection_points(skel, radius, center=None, _geometry=None):
    """Analytic transverse circle/segment intersections, returned as (x, y).

    Tangencies are excluded. At a shared pixel exactly on the ring, count one
    location if incident segments reach both sides. A terminal pixel exactly
    on the ring counts once if its incident segment is inside (half-open end
    convention). Isolated pixels have no crossings. Distinct crossings within
    the same former pixel annulus remain distinct, including two on one chord.
    """
    if not np.isfinite(radius) or radius <= 0:
        raise ValueError("Radius must be finite and positive.")
    if center is None:
        center = skeleton_centroid(np.asarray(skel, dtype=bool))
    if center[0] is None:
        return np.empty((0, 2))
    center = np.asarray(center, float)
    if center.shape != (2,) or not np.isfinite(center).all():
        raise ValueError("Center must contain finite x, y coordinates.")
    graph, nodes, xy, edges = _geometry if _geometry is not None else _sholl_geometry(skel)
    if not len(edges):
        return np.empty((0, 2))
    a = xy[edges[:, 0]] - center
    delta = xy[edges[:, 1]] - xy[edges[:, 0]]
    aa = np.einsum('ij,ij->i', delta, delta)
    bb = 2 * np.einsum('ij,ij->i', a, delta)
    cc = np.einsum('ij,ij->i', a, a) - radius**2
    discriminant = bb**2 - 4 * aa * cc
    # A zero discriminant is a tangent, not a crossing.
    good = discriminant > 1e-10
    roots = np.sqrt(np.maximum(discriminant, 0))
    points = []
    for sign in (-1, 1):
        t = (-bb + sign * roots) / (2 * aa)
        mask = good & (t > 1e-9) & (t < 1 - 1e-9)
        points.extend((a[mask] + t[mask, None] * delta[mask] + center).tolist())
    radial_delta = np.linalg.norm(xy - center, axis=1) - radius
    node_index = {node: i for i, node in enumerate(nodes)}
    for i in np.flatnonzero(np.abs(radial_delta) < 1e-9):
        neighbors = list(graph.neighbors(nodes[i]))
        sides = np.asarray([radial_delta[node_index[n]] for n in neighbors])
        inside = np.any(sides < -1e-9)
        outside = np.any(sides > 1e-9)
        if inside and (outside or len(neighbors) == 1):
            points.append(xy[i].tolist())
    if not points:
        return np.empty((0, 2))
    # A shared location at a junction is one intersection, not one per edge.
    return np.unique(np.round(np.asarray(points), 9), axis=0)

def sholl_analysis(skel, step=10, max_radius=None, center=None):
    """Sholl profile in pixels, centered on skeleton centroid by default.

    The final automatic ring lies strictly outside all foreground. Values
    beyond that support are zero, permitting valid batch-wide zero padding.
    AUC integrates only the sampled positive-radius interval; decay_rate is
    the ordinary linear slope of counts versus radius, not an exponential fit.
    """
    skel = np.asarray(skel, dtype=bool)
    if skel.ndim != 2 or not np.isfinite(step) or step <= 0:
        raise ValueError("Expected a 2-D skeleton and a positive finite step.")
    if max_radius is not None and (not np.isfinite(max_radius) or max_radius <= 0):
        raise ValueError("max_radius must be finite and positive.")
    cx, cy = skeleton_centroid(skel) if center is None else center
    empty = dict(radii=np.array([], float), intersections=np.array([], int),
                 max_intersections=0, radius_of_max=np.nan, auc=0.0,
                 decay_rate=np.nan, center_x=np.nan, center_y=np.nan)
    if not skel.any():
        return empty
    geometry = _sholl_geometry(skel)
    support = np.linalg.norm(geometry[2] - np.asarray([cx, cy]), axis=1).max()
    stop = (np.floor(support / step) + 1) * step if max_radius is None else max_radius
    radii = np.arange(1, int(np.floor(stop / step)) + 1) * step
    intersections = np.asarray([len(sholl_intersection_points(skel, r, (cx, cy), geometry))
                                for r in radii], int)
    empty.update(radii=radii, intersections=intersections, center_x=cx, center_y=cy)
    if len(radii):
        empty.update(max_intersections=int(intersections.max()),
                     radius_of_max=float(radii[intersections.argmax()]) if intersections.any() else np.nan,
                     auc=float(np.sum((intersections[1:] + intersections[:-1]) * np.diff(radii) / 2)),
                     decay_rate=float(np.polyfit(radii, intersections, 1)[0]) if len(radii) > 1 else np.nan)
    return empty


# ============================================================
# Batch Sholl Analysis Across All Microglial Samples
# ============================================================

sholl_rows = []

for path, cell_state in zip(all_paths, all_cell_states):
    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)

        sh = sholl_analysis(skel)

        sholl_rows.append({
            "filename": data["filename"],
            "sample_id": data["sample_id"],
            "cell_state": data["cell_state"],
            "max_intersections": sh["max_intersections"],
            "radius_of_max": sh["radius_of_max"],
            "auc": sh["auc"],
            "decay_rate": sh["decay_rate"],
            "radii": sh["radii"],
            "intersections": sh["intersections"]
        })

    except Exception as e:
        print("Error processing:", path, e)
        continue

df_sholl = pd.DataFrame(sholl_rows)

# ============================================================
# Sholl Wide Table (one column per radius)
# Required for Mixed ANOVA in Module 9
# ============================================================

# Collect all radii found across all cells
all_sholl_radii = sorted({
    int(r)
    for radii in df_sholl["radii"]
    for r in radii
})

sholl_wide_rows = []

for _, row in df_sholl.iterrows():
    entry = {
        "filename": row["filename"],
        "cell_state": row["cell_state"]
    }

    # Create a dictionary radius -> intersections
    radius_to_value = {
        int(r): float(v)
        for r, v in zip(row["radii"], row["intersections"])
    }

    # Use the same radius columns for every cell
    # Radii outside the complete skeleton support have zero intersections.
    for r in all_sholl_radii:
        entry[f"sholl_{r}"] = radius_to_value.get(r, 0.0)

    sholl_wide_rows.append(entry)

df_sholl_wide = pd.DataFrame(sholl_wide_rows)

# Uses dot decimal and comma separator.
df_sholl_wide.to_csv(
    os.path.join(EXPORT_DIR, "Module 8 - sholl_wide_table.csv"),
    index=False
)

df_sholl_wide

# ------------------------------------------------------------
# EXPORT TABLE (Batch Sholl Analysis Across All Microglial Samples)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_sholl.to_csv(os.path.join(EXPORT_DIR, "Module 8 - batch_sholl_analysis_across_all_microglial_samples.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_sholl

**Table 7. Full geometric Sholl profiles.**

For each unique image, the table reports its group, summary quantities and the full arrays of sampled radii and crossing counts. Counts come from circle–segment geometry with the tangency/shared-vertex conventions in Module 8, not from the number of skeleton pixels close to a ring. The first radius is 10 pixels and the automatic final ring lies outside the skeleton.


In [ ]:
# ============================================================
# Sholl Summary Table
# ============================================================

df_sholl_summary = df_sholl.drop(columns=["radii", "intersections"])

# ------------------------------------------------------------
# EXPORT TABLE (Sholl Summary Table)
# ------------------------------------------------------------

# Uses dot decimal and comma separator.
df_sholl_summary.to_csv(os.path.join(EXPORT_DIR, "Module 8 - sholl_summary_table.csv"), index=False)
# IMPORTANT: This saves the table above. Change the filename ONLY if needed.

df_sholl_summary

**Table 8. Scalar Sholl descriptors.**

The summary retains maximum crossings, radius of the first maximum, AUC over sampled positive radii and the linear count-versus-radius slope. AUC and slopes use preprocessed-pixel radii. decay_rate is a descriptive linear slope, not evidence of exponential process retraction. Full curves and radius-specific inference are supplied separately.


In [ ]:
metrics = ["max_intersections", "radius_of_max", "auc", "decay_rate"]

for metric in metrics:
    plot_bars_or_boxplot(
        df=df_sholl_summary,
        value_col=metric,
        group_col="cell_state",
        filename_col="sample_id",
        state_order=state_order,
        ylabel=metric.replace("_", " ").capitalize(),
        xlabel="Image Filename / Group",
        title=f"Sholl Metric — {metric.replace('_', ' ').capitalize()}",
        save_path=os.path.join(EXPORT_DIR, f"Module 8 - sholl_{metric}.png")
    )

**Figure 15. Distributions of Sholl summary descriptors.**

Panels compare maximum geometric crossings, radius of maximum, AUC and the linear profile slope across user-defined groups. Points or bars identify images; grouped boxplots summarize available observations when selected by the common plotting utility. Inferential results are reported in Module 9, rather than implied by apparent differences in these plots.


In [ ]:
# ============================================================
# Sholl Overlay for All Microglial Samples
# ============================================================

# Number of samples
n = len(all_paths)

# Dynamic grid layout
cols = 4
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten()

for i, (path, cell_state) in enumerate(zip(all_paths, all_cell_states)):

    try:
        data = preprocessed_data[path]

        skel = data["skeleton_visual_uint8"].astype(bool)

        sh = sholl_analysis(skel)

        radii = sh["radii"]

        cx, cy = sh["center_x"], sh["center_y"]

        if np.isnan(cx) or np.isnan(cy):
            continue

        radii_to_plot = radii[::5]
        point_sets = [sholl_intersection_points(skel, float(r), (cx, cy)) for r in radii_to_plot]
        intersection_points = (np.concatenate(point_sets, axis=0)
                               if point_sets else np.empty((0, 2)))

        ax = axes[i]
        ax.imshow(skel, cmap="gray")

        # Zoom around the skeleton so the cell is visible
        ys_skel, xs_skel = np.where(skel)

        if len(xs_skel) > 0:
            margin = 80

            x_min = max(xs_skel.min() - margin, 0)
            x_max = min(xs_skel.max() + margin, skel.shape[1])

            y_min = max(ys_skel.min() - margin, 0)
            y_max = min(ys_skel.max() + margin, skel.shape[0])

            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_max, y_min)

        ax.scatter([cx], [cy], c="yellow", s=40)

        theta = np.linspace(0, 2 * np.pi, 400)
        for r in radii_to_plot:
            x = cx + r * np.cos(theta)
            y = cy + r * np.sin(theta)
            ax.plot(x, y, color="cyan", alpha=0.15, linewidth=0.6)

        if len(intersection_points):
            ax.scatter(intersection_points[:, 0], intersection_points[:, 1], c="red", s=8)

        ax.set_title(f"{data['filename']} ({data['cell_state']})", fontsize=9)
        ax.axis("off")

    except Exception as e:
        print("Error processing:", path, e)
        continue

# Hide unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()

# ------------------------------------------------------------
# EXPORT FIGURE (Sholl Overlay for All Microglial Samples)
# ------------------------------------------------------------
plt.savefig(os.path.join(EXPORT_DIR, "Module 8 - sholl_overlay_for_all_microglial_samples.png"), dpi=600, bbox_inches="tight")
# IMPORTANT: This saves the figure above. Change the filename ONLY if needed.

plt.show() 

**Figure 16. Geometric Sholl overlays for visual assessment.**

The visual skeleton appears in grayscale, its centroid in yellow, selected rings in cyan and exact counted intersection locations in red. Only every fifth sampled ring is drawn to reduce clutter; the numerical profile uses all sampled rings. The centre is the skeleton centroid, not automatically the soma. Review boundary truncation, disconnected processes and segmentation before interpreting counts.


In [ ]:
# ============================================================
# Build Master Morphometric DataFrame
# ============================================================

# ============================================================
# Pandas display settings
# ============================================================
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# ============================================================
# Identify all feature dataframes available in the workspace
# ============================================================

feature_dfs = []

feature_candidates = [
    ("df_sholl_summary", globals().get("df_sholl_summary")),
    ("df_sholl_wide", globals().get("df_sholl_wide")),
    ("df_fd", globals().get("df_fd")),
    ("df_lac", globals().get("df_lac")),
    ("df_glcm", globals().get("df_glcm")),
    ("df_local_fd", globals().get("df_local_fd")),
    ("df_graph", globals().get("df_graph")),
]

for name, df in feature_candidates:
    if df is not None and not df.empty:
        feature_dfs.append(df.copy())

if len(feature_dfs) == 0:
    raise ValueError("No feature dataframes found. Please run Modules 1–8 first.")

# ============================================================
# Utility: detect ID and state columns
# ============================================================

def detect_id_column(df):
    # Prefer filename because it is shared by all feature tables.
    # sample_id is useful for display, but causes duplicate columns during merges.
    candidates = ["cell_id", "filename", "image_id", "image_name", "id", "sample_id"]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError("No ID column found in dataframe.")


def detect_state_column(df):
    candidates = ["cell_state", "state", "group", "label", "phenotype"]

    for c in candidates:
        if c in df.columns:
            return c

    raise KeyError("No cell_state column found in dataframe.")

# ============================================================
# Standardize column names and remove auxiliary duplicate IDs
# ============================================================

clean_dfs = []
sample_id_maps = []

for df in feature_dfs:
    df = df.copy()

    id_col = detect_id_column(df)
    state_col = detect_state_column(df)

    df.rename(columns={id_col: "cell_id", state_col: "cell_state"}, inplace=True)

    # Save sample_id once, only for final display
    if "sample_id" in df.columns:
        tmp_sample = df[["cell_id", "cell_state", "sample_id"]].copy()
        tmp_sample = tmp_sample.dropna(subset=["cell_id", "cell_state"])
        tmp_sample = tmp_sample.drop_duplicates(subset=["cell_id", "cell_state"])
        sample_id_maps.append(tmp_sample)

    # Remove auxiliary ID/display columns before merging
    # They cause sample_id_x / sample_id_y conflicts.
    auxiliary_id_cols = ["sample_id", "filename", "image_id", "image_name", "id"]

    for col in auxiliary_id_cols:
        if col in df.columns and col not in ["cell_id", "cell_state"]:
            df.drop(columns=[col], inplace=True)

    # Remove duplicated columns inside the same dataframe
    df = df.loc[:, ~df.columns.duplicated()]

    clean_dfs.append(df)

# ============================================================
# Merge all feature tables on cell_id + cell_state
# ============================================================

df_master = clean_dfs[0]

for df in clean_dfs[1:]:

    # Safety: remove repeated non-key columns before merging
    overlapping_cols = [
        c for c in df.columns
        if c in df_master.columns and c not in ["cell_id", "cell_state"]
    ]

    if overlapping_cols:
        df = df.drop(columns=overlapping_cols)

    df_master = df_master.merge(
        df,
        on=["cell_id", "cell_state"],
        how="outer"
    )

# ============================================================
# Add one clean sample_id column back, if available
# ============================================================

if len(sample_id_maps) > 0:
    df_sample_id = pd.concat(sample_id_maps, ignore_index=True)
    df_sample_id = df_sample_id.drop_duplicates(subset=["cell_id", "cell_state"])

    df_master = df_master.merge(
        df_sample_id,
        on=["cell_id", "cell_state"],
        how="left"
    )

# Remove duplicate columns as final safety
df_master = df_master.loc[:, ~df_master.columns.duplicated()]

# Sort columns
priority_cols = ["cell_id", "sample_id", "cell_state"]

cols = [c for c in priority_cols if c in df_master.columns] + \
       [c for c in df_master.columns if c not in priority_cols]

df_master = df_master[cols]

# ============================================================
# EXPORT
# ============================================================

# Uses dot decimal and comma separator.
df_master.to_csv(os.path.join(EXPORT_DIR, "Module 8 - build_master_morphometric_dataframe_universal.csv"), index=False)

df_master

print("\n[INFO] ===== Module 8 Completed =====")

**Table 9. Universal morphometric dataframe.**

One row represents one uniquely identified image/cell. cell_id is derived from the relative image path, with the group retained separately. The matrix combines scalar descriptors and radius-specific Sholl counts with corrected MFS slopes (local_fd_scale_*). junction_pixel_count replaces bifurcations and total_length is absent. Metadata/identifier fields are excluded from model predictors; constant and entirely missing predictors are also excluded at model preparation.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


## Module 9 — Statistical Testing

The input is df_master. Scalar descriptors are compared between cell_state groups. Columns sholl_<radius> form a separate repeated-measures branch. Columns local_fd_scale_* are excluded from scalar tests; no additional inferential MFS-profile model is implemented.

### Scalar decision tree

Within each group, fewer than three observations or zero variance makes normality unevaluable; otherwise Shapiro–Wilk is used below 5000 observations and Lilliefors at 5000 or more. All groups must pass the selected normality check to enter the parametric branch. Levene's median-centred test then assesses variance homogeneity.

| Groups and assumptions | Global test | Effect size |
|---|---|---|
| Two groups, normal/equal variance | Student t | Cohen's d |
| Two groups, normal/unequal variance | Welch t | Cohen's d |
| Two groups, nonnormal or unevaluable | Mann–Whitney U | Rank-biserial correlation |
| More than two, normal/equal variance | ANOVA | Eta squared |
| More than two, normal/unequal variance | Welch ANOVA | Partial eta squared when returned |
| More than two, nonnormal or unevaluable | Kruskal–Wallis | Epsilon squared |

Globally constant or unusable descriptors are skipped. Automated test selection does not establish that a small dataset meets every inferential assumption.

### Mixed ANOVA for Sholl

The model uses cell_state between subjects, numeric radius within subjects and the globally unique cell_id as subject. At least two sampled radii and two complete subjects per group are required in at least two groups. Incomplete subjects are removed explicitly and counted. Radii constant across all retained subjects are omitted from inference, listed in the export and retained in descriptive tables.

Three effects are exported individually: **Sholl_between, Sholl_radius and Sholl_interaction**. The interaction is selected by its actual Source label, not by searching for an asterisk or falling back to the first row.

**Version and extraction.** This release fixes pingouin==0.7.0. W_spher and p_spher are obtained by field name from the mixed-ANOVA table. The former mauchly[1] extraction was incorrect because that position is W rather than p. epsilon(..., correction='gg') is read as a scalar on group-centred residual profiles and checked against the mixed-model epsilon.

**Sphericity corrections.** GG and HF corrections multiply both numerator and denominator degrees of freedom for the radius and interaction effects. HF uses the mixed-design formula with N subjects, g groups and d = number of radii − 1:

HF = min(1, (N*d*GG − 2) / (d*(N − g − d*GG))), limited to [1/d, 1]. If the denominator is not positive, the upper limit 1 is used. GG/HF corrected p-values are computed from the relevant mixed-model F and corrected degrees of freedom.

The **predefined primary p_value is GG-corrected for radius and interaction**, and uncorrected for the between-group effect, which does not require sphericity. HF is exported additionally; the code does not select whichever p is smaller. Both raw and corrected values remain available.

With two radii, sphericity holds automatically. If the contrast matrix is rank deficient, Mauchly's W/p are marked unavailable with a status flag; the code does not interpret a singular test as evidence that sphericity holds. Estimable GG/HF quantities remain available. An unestimable model is recorded with an explicit error and missing p, rather than a fabricated result.

The export distinguishes n_subjects_input, n_subjects_excluded, n_total (analyzed subjects), n_observations and n_radii. It also includes F, original and GG/HF degrees of freedom, p_unc, p_GG, p_HF, primary-p correction, partial eta squared, W, p of Mauchly and epsilons.

### Multiplicity and post-hoc tests

BH/FDR is calculated globally (p_adj_global) and separately within descriptor families (p_adj_family). **p_adj aliases the family-adjusted result**, not the global result. For Sholl, multiplicity adjustment acts on the primary p already defined above. Sphericity correction and multiplicity correction are distinct operations.

Scalar metrics significant after family FDR and compared across more than two groups may receive Tukey HSD after ANOVA, Games–Howell after Welch ANOVA, or Dunn with BH after Kruskal–Wallis. Sholl effects and MFS scale columns do not receive those scalar post-hoc comparisons. Raw significance, family FDR and global FDR flags are exported separately.

### Experimental unit and limits

The current subject is an image/cell, not an animal or patient. Multiple cells from the same animal/patient require a design that models that dependence; such metadata are not inferred from the image. Counts, small samples and asymmetric profiles may require a different model. Synthetic examples exercise the code but do not validate biological hypotheses or experimental independence.


In [ ]:
def sholl_mixed_anova(df, sholl_cols, group_col="cell_state"):
    """Export all three effects, raw/GG/HF p, corrected df and diagnostics.

    Requires Pingouin 0.7.0, which fixes pooled covariance and the mixed-model
    GG corrections. GG is the pre-specified primary p for within and interaction;
    between-group inference uses the uncorrected p. FDR is applied afterwards.
    HF uses the mixed-design residual df N-g, not the one-way N-1 formula.
    """
    if pg.__version__ != "0.7.0":
        raise ValueError("This validated analysis requires pingouin==0.7.0.")
    cols = sorted(sholl_cols, key=lambda c: float(c.removeprefix("sholl_")))
    if len(cols) < 2:
        raise ValueError("At least two sampled radii are required.")
    if df['cell_id'].isna().any() or df[group_col].isna().any():
        raise ValueError("Subject IDs and group labels cannot be missing.")
    if df['cell_id'].duplicated().any():
        raise ValueError("Each cell_id must be globally unique across groups.")
    selected = df[['cell_id', group_col] + cols].copy()
    selected[cols] = selected[cols].replace([np.inf, -np.inf], np.nan)
    n_input = len(selected)
    selected = selected.dropna(subset=cols)  # Explicit complete-subject analysis.
    n, g = len(selected), selected[group_col].nunique()
    if g < 2 or (selected.groupby(group_col).size() < 2).any():
        raise ValueError("Mixed ANOVA requires at least two complete cells in each of two groups.")
    # Globally constant zero outer rings do not carry radial information and
    # make the contrast covariance singular. Exclude them from inference only.
    excluded = [c for c in cols if selected[c].nunique() <= 1]
    cols = [c for c in cols if c not in excluded]
    if len(cols) < 2:
        raise ValueError("Fewer than two nonconstant radii remain for inference.")
    wide = selected.set_index('cell_id')[cols]
    labels = selected.set_index('cell_id')[group_col]
    residual = wide - wide.groupby(labels, observed=True).transform('mean')
    # Public epsilon returns one scalar. Mauchly's p and W are obtained from
    # the corrected mixed-ANOVA table, not from a tuple positional index.
    gg = float(pg.epsilon(residual, correction='gg'))
    d = len(cols) - 1
    denom = d * (n - g - d * gg)
    hf = float(np.clip((n * d * gg - 2) / denom, 1 / d, 1)) if denom > 0 else 1.0
    long = selected[['cell_id', group_col] + cols].melt(
        id_vars=['cell_id', group_col], var_name='radius', value_name='value')
    long['radius_num'] = long['radius'].str.removeprefix('sholl_').astype(float)
    mix = pg.mixed_anova(data=long, dv='value', within='radius_num',
                         between=group_col, subject='cell_id', correction=True)
    within = mix.loc[mix['Source'] == 'radius_num'].iloc[0]
    if not np.isfinite(gg) or not np.isclose(gg, within['eps'], rtol=1e-8):
        raise ValueError("Invalid or inconsistent pooled Greenhouse-Geisser epsilon.")
    # Rank deficiency beyond the harmless removed constant rings invalidates
    # Mauchly's chi-square approximation. Keep GG/HF; flag Mauchly unavailable.
    from scipy.linalg import helmert
    contrast = residual.to_numpy() @ helmert(len(cols)).T
    rank_ok = np.linalg.matrix_rank(contrast) == d
    sph_p = float(within.get('p_spher', 1.0 if d == 1 else np.nan)) if rank_ok else np.nan
    sph_w = float(within.get('W_spher', np.nan)) if rank_ok else np.nan
    rows = []
    effects = {group_col: 'between', 'radius_num': 'radius', 'Interaction': 'interaction'}
    for _, row in mix.iterrows():
        effect = effects[row['Source']]
        repeated = effect != 'between'
        df1, df2, F = float(row['DF1']), float(row['DF2']), float(row['F'])
        p_unc = float(row['p_unc'])
        p_gg = float(f.sf(F, df1 * gg, df2 * gg)) if repeated else np.nan
        p_hf = float(f.sf(F, df1 * hf, df2 * hf)) if repeated else np.nan
        if repeated and not np.isclose(p_gg, row.get('p_GG_corr', p_unc if d == 1 else np.nan), rtol=1e-8, atol=1e-14):
            raise ValueError("GG p-value does not match the mixed-model F and df.")
        rows.append(dict(
            metric='Sholl_' + effect, effect=effect, n_groups=g, n_total=n,
            n_observations=n * len(cols), n_subjects_input=n_input,
            n_subjects_excluded=n_input - n, n_radii=len(cols),
            excluded_constant_radii=' | '.join(excluded),
            groups=' | '.join(map(str, selected[group_col].unique())),
            normal=np.nan, normality_method='mixed ANOVA; assumptions require review',
            normality_p_by_group=np.nan, levene_p=np.nan, equal_variance=np.nan,
            test='Mixed ANOVA (between x radius)', statistic=F,
            DF1=df1, DF2=df2, p_unc=p_unc, p_GG=p_gg, p_HF=p_hf,
            p_value=p_gg if repeated else p_unc,
            p_value_correction='Greenhouse-Geisser' if repeated else 'none',
            effect_size=float(row['np2']), effect_size_name='np2',
            sphericity_W=sph_w if repeated else np.nan,
            sphericity_p=sph_p if repeated else np.nan,
            sphericity_status=('automatic_two_radii' if d == 1 else
                              'estimated' if rank_ok else 'unavailable_rank_deficient') if repeated else 'not_applicable',
            GG_epsilon=gg if repeated else np.nan, HF_epsilon=hf if repeated else np.nan,
            DF1_GG=df1 * gg if repeated else np.nan, DF2_GG=df2 * gg if repeated else np.nan,
            DF1_HF=df1 * hf if repeated else np.nan, DF2_HF=df2 * hf if repeated else np.nan))
    return rows


# ============================================================
# Module 9 — Statistical Testing
# ============================================================
# This module performs:
#   - Between-group ANOVA / Welch / Kruskal–Wallis
#   - Mixed ANOVA (between × within) for Sholl metrics
#   - Mauchly’s test of sphericity
#   - Greenhouse–Geisser / Huynh–Feldt corrections
#
# SPECIALIZED FOR MICROGLIA:
#   - Uses cell_state as grouping factor
#   - Optimized for small datasets
#   - Compatible with df_master (universal feature table)
# ============================================================
# BETWEEN-GROUP PIPELINE
#   Normality:
#       - n < 3     -> insufficient_n -> treated as non-parametric
#       - 3 <= n < 5000  -> Shapiro-Wilk
#       - n >= 5000      -> Lilliefors
#       - zero variance  -> zero_variance -> treated as non-parametric
#
#   If normal:
#       - Levene for variance homogeneity
#       - 2 groups:
#           * equal variances    -> Student t-test
#           * unequal variances  -> Welch t-test
#       - >2 groups:
#           * equal variances    -> ANOVA
#           * unequal variances  -> Welch ANOVA
#
#   If non-normal:
#       - 2 groups  -> Mann–Whitney U
#       - >2 groups -> Kruskal–Wallis
#
# EFFECT SIZES
#   - Student/Welch t-test -> Cohen's d
#   - Mann–Whitney U       -> rank-biserial correlation (RBC)
#   - ANOVA                -> eta squared
#   - Welch ANOVA          -> np2 if available from pingouin
#   - Kruskal–Wallis       -> epsilon squared
#
# MULTIPLE TESTING
#   - Global p-values across metrics -> FDR Benjamini-Hochberg
#
# REPEATED MEASURES
#   - Sholl metrics handled separately with mixed ANOVA
# ============================================================

print("\n[INFO] ===== Module 9 - Statistical Testing =====")

warnings.filterwarnings("default")

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

if "df_master" not in globals():
    raise NameError("df_master not found. Please run the Master Builder module first.")

df_stats = df_master.copy()

GROUP_COL = "cell_state"
if GROUP_COL not in df_stats.columns:
    raise KeyError("cell_state column not found in df_master.")

if "cell_id" not in df_stats.columns:
    raise KeyError("cell_id column not found in df_master.")

# ------------------------------------------------------------
# Numeric metrics
# ------------------------------------------------------------
numeric_cols = [
    c for c in df_stats.select_dtypes(include=[np.number]).columns
    if c not in ["cell_id"]
]

# Repeated-measures patterns excluded from between-group tests
rm_patterns = ["sholl_", "local_fd_scale_"]

def is_repeated_measures_metric(metric):
    return any(p in metric.lower() for p in rm_patterns)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def clean_group_data(df, metric, group_col):
    """
    Returns:
        valid_groups: list of group labels with at least 1 non-NA value
        data_by_group: dict[group] -> 1D numpy array
        n_groups: number of valid groups
        n_total: total non-missing observations
    """
    sub = df[[group_col, metric]].dropna().copy()
    valid_groups = []
    data_by_group = {}

    for g in sub[group_col].dropna().unique():
        vals = sub.loc[sub[group_col] == g, metric].dropna().values.astype(float)
        if len(vals) > 0:
            valid_groups.append(g)
            data_by_group[g] = vals

    return valid_groups, data_by_group, len(valid_groups), len(sub)

def group_normality_test(x):
    """
    Normality per group:
      - n < 3: cannot reliably test
      - zero variance: not evaluable
      - 3 <= n < 5000: Shapiro-Wilk
      - n >= 5000: Lilliefors
    """
    n = len(x)

    if n < 3:
        return np.nan, "insufficient_n"

    if np.nanstd(x) == 0:
        return np.nan, "zero_variance"

    if n < 5000:
        stat, p = shapiro(x)
        return p, "Shapiro-Wilk"
    else:
        stat, p = lilliefors(x, dist="norm")
        return p, "Lilliefors"

def evaluate_normality(data_by_group, alpha=0.05):
    """
    A metric is considered normal only if ALL groups:
      - have testable sample size (>= 3)
      - do not have zero variance
      - pass the relevant normality test
    """
    pvals = {}
    methods = {}
    normal = True

    for g, x in data_by_group.items():
        p, method = group_normality_test(x)
        pvals[g] = p
        methods[g] = method

        if np.isnan(p):
            normal = False
        elif p <= alpha:
            normal = False

    return normal, pvals, methods

def safe_eta_squared_from_anova(F_value, k, n_total):
    """
    eta^2 from one-way ANOVA F statistic
    """
    try:
        df_between = k - 1
        df_within = n_total - k
        if df_within <= 0:
            return np.nan
        return (F_value * df_between) / ((F_value * df_between) + df_within)
    except Exception:
        return np.nan

def safe_epsilon_squared_from_kruskal(H_value, k, n_total):
    """
    epsilon^2 for Kruskal-Wallis
    """
    try:
        denom = n_total - k
        if denom <= 0:
            return np.nan
        return (H_value - k + 1) / denom
    except Exception:
        return np.nan

# ------------------------------------------------------------
# 1. BETWEEN-GROUP TESTS
# ------------------------------------------------------------
results = []

for metric in numeric_cols:

    if is_repeated_measures_metric(metric):
        continue

    valid_groups, data_by_group, k, n_total = clean_group_data(df_stats, metric, GROUP_COL)

    # Need at least 2 groups with data
    if k < 2:
        warnings.warn(f"[{metric}] skipped: fewer than 2 groups with data.")
        continue

    # Avoid zero-variance total metrics that cannot be tested meaningfully
    try:
        all_values = np.concatenate([data_by_group[g] for g in valid_groups])
    except Exception as e:
        warnings.warn(f"[{metric}] skipped: could not concatenate values ({e}).")
        continue

    if len(all_values) == 0 or np.nanstd(all_values) == 0:
        warnings.warn(f"[{metric}] skipped: zero global variance or no valid values.")
        continue

    # ----------------------------
    # Normality
    # ----------------------------
    normal, norm_pvals, norm_methods = evaluate_normality(data_by_group, alpha=0.05)

    normality_method_summary = "; ".join(
        [f"{g}:{norm_methods[g]}" for g in valid_groups]
    )
    normality_p_summary = "; ".join(
        [f"{g}:{'nan' if pd.isna(norm_pvals[g]) else round(norm_pvals[g], 6)}" for g in valid_groups]
    )

    # ----------------------------
    # If normal -> Levene
    # ----------------------------
    lev_p = np.nan
    equal_var = np.nan

    if normal:
        try:
            lev_stat, lev_p = levene(*[data_by_group[g] for g in valid_groups], center="median")
            equal_var = bool(lev_p > 0.05)
        except Exception as e:
            warnings.warn(f"[{metric}] Levene test failed: {e}")
            lev_p = np.nan
            equal_var = False

    # ----------------------------
    # Choose test based on k
    # ----------------------------
    test_name = None
    stat = np.nan
    p = np.nan
    effect_size = np.nan
    effect_size_name = np.nan

    try:
        # ====================================================
        # TWO GROUPS
        # ====================================================
        if k == 2:
            g1, g2 = valid_groups
            x1 = data_by_group[g1]
            x2 = data_by_group[g2]

            if normal:
                if equal_var:
                    test_name = "Student t-test"
                    stat, p = ttest_ind(x1, x2, equal_var=True, nan_policy="omit")
                else:
                    test_name = "Welch t-test"
                    stat, p = ttest_ind(x1, x2, equal_var=False, nan_policy="omit")

                try:
                    effect_size = pg.compute_effsize(x1, x2, eftype="cohen")
                    effect_size_name = "cohen_d"
                except Exception as e:
                    warnings.warn(f"[{metric}] effect size computation failed: {e}")
                    effect_size = np.nan
                    effect_size_name = np.nan

            else:
                test_name = "Mann–Whitney U"
                stat, p = mannwhitneyu(x1, x2, alternative="two-sided")

                try:
                    mw = pg.mwu(x1, x2, alternative="two-sided")
                    if "RBC" in mw.columns:
                        effect_size = mw["RBC"].iloc[0]
                        effect_size_name = "rank_biserial_correlation"
                    else:
                        warnings.warn(
                            f"[{metric}] Mann–Whitney effect size column RBC not found. "
                            f"Available columns: {list(mw.columns)}"
                        )
                        effect_size = np.nan
                        effect_size_name = np.nan
                except Exception as e:
                    warnings.warn(f"[{metric}] Mann–Whitney effect size failed: {e}")
                    effect_size = np.nan
                    effect_size_name = np.nan

        # ====================================================
        # MORE THAN TWO GROUPS
        # ====================================================
        elif k > 2:
            arrays = [data_by_group[g] for g in valid_groups]

            if normal:
                if equal_var:
                    test_name = "ANOVA"
                    stat, p = f_oneway(*arrays)

                    try:
                        effect_size = safe_eta_squared_from_anova(stat, k, n_total)
                        effect_size_name = "eta_squared"
                    except Exception as e:
                        warnings.warn(f"[{metric}] ANOVA effect size failed: {e}")
                        effect_size = np.nan
                        effect_size_name = np.nan

                else:
                    test_name = "Welch ANOVA"
                    tmp = df_stats[[GROUP_COL, metric]].dropna().copy()
                    welch = pg.welch_anova(dv=metric, between=GROUP_COL, data=tmp)

                    # Normalize column names across pingouin versions
                    welch_cols_map = {c: c.lower().replace("-", "_") for c in welch.columns}

                    # Statistic
                    stat_found = False
                    for original_col, norm_col in welch_cols_map.items():
                        if norm_col in ["f", "f_val"]:
                            stat = welch[original_col].iloc[0]
                            stat_found = True
                            break

                    if not stat_found:
                        warnings.warn(
                            f"[{metric}] Welch ANOVA statistic extraction failed. "
                            f"Available columns: {list(welch.columns)}"
                        )

                    # p-value
                    p_found = False
                    for original_col, norm_col in welch_cols_map.items():
                        if norm_col in ["p_unc", "p", "pval", "p_value"]:
                            p = welch[original_col].iloc[0]
                            p_found = True
                            break

                    if not p_found or pd.isna(p):
                        warnings.warn(
                            f"[{metric}] Welch ANOVA p-value extraction failed. "
                            f"Available columns: {list(welch.columns)}"
                        )

                    # effect size
                    es_found = False
                    for original_col, norm_col in welch_cols_map.items():
                        if norm_col in ["np2", "eta2", "eta_squared"]:
                            effect_size = welch[original_col].iloc[0]
                            effect_size_name = norm_col
                            es_found = True
                            break

                    if not es_found:
                        warnings.warn(
                            f"[{metric}] Welch ANOVA effect size not found. "
                            f"Available columns: {list(welch.columns)}"
                        )
                        effect_size = np.nan
                        effect_size_name = np.nan

            else:
                test_name = "Kruskal–Wallis"
                stat, p = kruskal(*arrays)

                try:
                    effect_size = safe_epsilon_squared_from_kruskal(stat, k, n_total)
                    effect_size_name = "epsilon_squared"
                except Exception as e:
                    warnings.warn(f"[{metric}] Kruskal effect size failed: {e}")
                    effect_size = np.nan
                    effect_size_name = np.nan

        # General warning if any test ran but p is still NaN
        if test_name is not None and pd.isna(p):
            warnings.warn(
                f"[{metric}] {test_name} returned p_value = NaN. "
                f"Check input data, group sizes, zero variance within groups, or library output format."
            )

    except Exception as e:
        test_name = "FAILED"
        stat = np.nan
        p = np.nan
        effect_size = np.nan
        effect_size_name = np.nan
        warnings.warn(f"[{metric}] test failed: {e}")

    results.append({
        "metric": metric,
        "n_groups": k,
        "n_total": n_total,
        "groups": " | ".join(map(str, valid_groups)),
        "normal": normal,
        "normality_method": normality_method_summary,
        "normality_p_by_group": normality_p_summary,
        "levene_p": lev_p,
        "equal_variance": equal_var,
        "test": test_name,
        "statistic": stat,
        "p_value": p,
        "effect_size": effect_size,
        "effect_size_name": effect_size_name
    })

# ------------------------------------------------------------
# 2. MIXED ANOVA FOR SHOLL
# ------------------------------------------------------------
sholl_cols = [c for c in numeric_cols if re.fullmatch(r"sholl_\d+(?:\.\d+)?", c)]
if len(sholl_cols) >= 2:
    try:
        results.extend(sholl_mixed_anova(df_stats, sholl_cols, GROUP_COL))
    except (ValueError, ArithmeticError, AssertionError) as exc:
        warnings.warn(f"[Sholl] Mixed ANOVA not estimable: {exc}")
        results.append({"metric": "Sholl_failed", "test": "Mixed ANOVA not estimable",
                        "error": str(exc), "p_value": np.nan})

# ------------------------------------------------------------
# 3. MULTIPLE TESTING CORRECTION BY METRIC FAMILY (BH/FDR)
# ------------------------------------------------------------
df_stats_results = pd.DataFrame(results)

def assign_metric_family(metric):
    """
    Assign each metric to a biologically interpretable family.
    FDR correction will be performed separately within each family.
    """
    m = str(metric).lower()

    # Sholl summary metrics
    if (
        m == "sholl"
        or "sholl" in m
        or m in ["max_intersections", "radius_of_max", "auc", "decay_rate"]
    ):
        return "sholl"

    # Fractal / lacunarity morphology
    if m in ["fractal_dimension", "lacunarity_mean"]:
        return "fractal_lacunarity"

    # Texture features
    if m in ["contrast", "homogeneity", "energy", "correlation"]:
        return "glcm_texture"

    # Skeleton / graph morphology
    if m in [
        "endpoints",
        "junction_pixel_count",
        "total_nodes",
        "total_edges",
        "total_edges",
        "diameter",
        "avg_degree",
        "avg_junction_degree",
        "endpoint_density",
        "junction_pixel_density",
        "junction_to_endpoint_ratio",
        "largest_component_nodes",
        "n_components"
    ]:
        return "graph_morphology"

    # Multiscale fractal spectrum
    if m.startswith("local_fd_scale_"):
        return "local_fd"

    # Fallback
    return "other"


if df_stats_results.empty:
    warnings.warn("No statistical results were generated.")
    df_stats_results["metric_family"] = np.nan
    df_stats_results["p_adj_global"] = np.nan
    df_stats_results["p_adj_family"] = np.nan
    df_stats_results["significant_global_fdr"] = False
    df_stats_results["significant_family_fdr"] = False

else:
    # Assign metric family
    df_stats_results["metric_family"] = df_stats_results["metric"].apply(assign_metric_family)

    valid_mask = df_stats_results["p_value"].notna()

    # --------------------------------------------------------
    # Optional: keep global FDR for reference
    # --------------------------------------------------------
    df_stats_results["p_adj_global"] = np.nan

    if valid_mask.any():
        df_stats_results.loc[valid_mask, "p_adj_global"] = multipletests(
            df_stats_results.loc[valid_mask, "p_value"],
            method="fdr_bh"
        )[1]
    else:
        warnings.warn("No valid p-values available for global BH/FDR correction.")

    # --------------------------------------------------------
    # Main correction: FDR within each metric family
    # --------------------------------------------------------
    df_stats_results["p_adj_family"] = np.nan

    for family, idx in df_stats_results[valid_mask].groupby("metric_family").groups.items():
        idx = list(idx)

        df_stats_results.loc[idx, "p_adj_family"] = multipletests(
            df_stats_results.loc[idx, "p_value"],
            method="fdr_bh"
        )[1]

    # Keep p_adj as the main column used downstream
    # Now p_adj means family-wise FDR-adjusted p-value
    df_stats_results["p_adj"] = df_stats_results["p_adj_family"]

    # Significance flags
    df_stats_results["significant_raw"] = df_stats_results["p_value"] < 0.05
    df_stats_results["significant_global_fdr"] = df_stats_results["p_adj_global"] < 0.05
    df_stats_results["significant_family_fdr"] = df_stats_results["p_adj_family"] < 0.05

    n_nan = (~valid_mask).sum()
    if n_nan > 0:
        warnings.warn(
            f"{n_nan} metrics had NaN p-values and were excluded from BH/FDR correction."
        )

# ------------------------------------------------------------
# Export
# ------------------------------------------------------------
df_stats_results.to_csv(os.path.join(EXPORT_DIR, "Module 9 - microglia_statistical_testing_results.csv"), index=False)

pd.set_option("display.max_rows", None)
df_stats_results

**Table 10. Scalar and mixed-ANOVA statistical results.**

Rows identify the descriptor or the specific Sholl effect. p_value is the predefined primary p: GG for Sholl radius/interaction and uncorrected for the between-group effect. Separate fields retain raw/GG/HF p-values and degrees of freedom. p_adj_family and p_adj_global represent different multiplicity scopes; p_adj uses the family result. Mauchly status, subject counts and excluded constant radii make limitations visible. Missing p-values remain missing and are excluded from FDR.


In [ ]:
# ============================================================
# Post-hoc Pairwise Comparisons
# ============================================================

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

if "df_master" not in globals():
    raise NameError("df_master not found. Please run Module 8 or the Master Builder first.")

if "df_stats_results" not in globals():
    raise NameError("df_stats_results not found. Please run the previous statistics module first.")

df_stats = df_master.copy()

GROUP_COL = "cell_state"
if GROUP_COL not in df_stats.columns:
    raise KeyError("cell_state column not found in df_master.")

def is_repeated_measures_metric(metric):
    return any(p in str(metric).lower() for p in ["sholl_", "local_fd_scale_"]) or str(metric).startswith("Sholl_")

def normalize_test_name(x):
    return str(x).strip().lower().replace("–", "-").replace("—", "-")

pairwise_rows = []

for _, row in df_stats_results.iterrows():

    metric = row["metric"]
    global_test = row["test"]
    p_adj_family = row.get("p_adj_family", row.get("p_adj", np.nan))
    metric_family = row.get("metric_family", "unknown")
    n_groups = row.get("n_groups", np.nan)

    # Skip repeated-measures metrics
    if is_repeated_measures_metric(metric):
        continue

    # Metric must exist in df_master
    if metric not in df_stats.columns:
        continue

    # Post-hoc only makes sense for >2 groups
    if pd.isna(n_groups) or int(n_groups) <= 2:
        continue

    # Only metrics significant after BH/FDR correction within their metric family
    if pd.isna(p_adj_family) or p_adj_family >= 0.05:
        continue

    sub = df_stats[[GROUP_COL, metric]].dropna().copy()
    if sub.empty:
        continue

    if sub[GROUP_COL].nunique() < 2:
        continue

    values = sub[metric].values
    labels = sub[GROUP_COL].values
    test_clean = normalize_test_name(global_test)

    # --------------------------------------------------------
    # ANOVA -> Tukey HSD
    # --------------------------------------------------------
    if test_clean == "anova":
        try:
            tukey = pairwise_tukeyhsd(endog=values, groups=labels, alpha=0.05)

            for res in tukey.summary().data[1:]:
                g1, g2, meandiff, p_adj, lower, upper, reject = res

                pairwise_rows.append({
                    "metric": metric,
                    "metric_family": metric_family,
                    "global_test": global_test,
                    "group1": g1,
                    "group2": g2,
                    "mean_difference": meandiff,
                    "test": "Tukey HSD",
                    "p_unc": np.nan,
                    "p_adj": p_adj,
                    "significant": bool(reject)
                })

        except Exception as e:
            print(f"[POSTHOC ERROR] metric={metric} | test={global_test} | Tukey failed: {e}")

    # --------------------------------------------------------
    # Welch ANOVA -> Games-Howell
    # --------------------------------------------------------
    elif test_clean == "welch anova":
        try:
            gh = pg.pairwise_gameshowell(data=sub, dv=metric, between=GROUP_COL)

            pcol = "pval" if "pval" in gh.columns else ("p_unc" if "p_unc" in gh.columns else ("p-unc" if "p-unc" in gh.columns else None))
            diffcol = "diff" if "diff" in gh.columns else None

            for _, r in gh.iterrows():
                pval = r[pcol] if pcol is not None else np.nan
                mdiff = r[diffcol] if diffcol is not None else np.nan

                pairwise_rows.append({
                    "metric": metric,
                    "metric_family": metric_family,
                    "global_test": global_test,
                    "group1": r["A"],
                    "group2": r["B"],
                    "mean_difference": mdiff,
                    "test": "Games-Howell",
                    "p_unc": np.nan,
                    "p_adj": pval,
                    "significant": bool(pval < 0.05) if pd.notna(pval) else False
                })

        except Exception as e:
            print(f"[POSTHOC ERROR] metric={metric} | test={global_test} | Games-Howell failed: {e}")

    # --------------------------------------------------------
    # Kruskal-Wallis -> Dunn + BH
    # --------------------------------------------------------
    elif test_clean == "kruskal-wallis":
        try:
            # Unadjusted Dunn
            dunn_unc = sp.posthoc_dunn(
                sub,
                val_col=metric,
                group_col=GROUP_COL,
                p_adjust=None
            )

            # BH-adjusted Dunn
            dunn_adj = sp.posthoc_dunn(
                sub,
                val_col=metric,
                group_col=GROUP_COL,
                p_adjust="fdr_bh"
            )

            groups = list(dunn_adj.index)

            for i in range(len(groups)):
                for j in range(i + 1, len(groups)):
                    g1 = groups[i]
                    g2 = groups[j]

                    p_unc = dunn_unc.loc[g1, g2]
                    p_adj = dunn_adj.loc[g1, g2]

                    mean_diff = (
                        sub.loc[sub[GROUP_COL] == g1, metric].mean()
                        - sub.loc[sub[GROUP_COL] == g2, metric].mean()
                    )

                    pairwise_rows.append({
                        "metric": metric,
                        "metric_family": metric_family,
                        "global_test": global_test,
                        "group1": g1,
                        "group2": g2,
                        "mean_difference": mean_diff,
                        "test": "Dunn (BH)",
                        "p_unc": p_unc,
                        "p_adj": p_adj,
                        "significant": bool(p_adj < 0.05) if pd.notna(p_adj) else False
                    })

        except Exception as e:
            print(f"[POSTHOC ERROR] metric={metric} | test={global_test} | Dunn failed: {e}")

df_posthoc = pd.DataFrame(pairwise_rows)

df_posthoc.to_csv(os.path.join(EXPORT_DIR, "Module 9 - posthoc_pairwise_comparisons.csv"), index=False)

pd.set_option("display.max_rows", None)

df_posthoc

print("\n[INFO] ===== Module 9 Completed =====")

**Table 11. Eligible scalar post-hoc comparisons.**

Comparisons are generated only for scalar metrics with more than two groups that pass the family-level FDR threshold. The selected procedures are Tukey HSD, Games–Howell, or Dunn with BH, according to the preceding global test. Radius/interaction effects and MFS scale columns are excluded from this scalar procedure. The table reports metric family, global test, compared groups, test-specific mean difference and adjusted p. No pairwise result is inferred merely from separation in a plot.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


### Module 10 — Dimensionality Reduction and Feature Importance

The module selects numeric predictors from the master matrix, excluding identifiers, legacy descriptor names, constant columns and completely missing columns. It replaces nonfinite values with missing values and imputes remaining missing entries with the column median. Corrected MFS slopes remain predictors under the internal local_fd_scale_* names; total_length and bifurcations are not reinstated.

The prepared matrix is standardized for PCA, UMAP and a Random Forest classifier with 500 trees, balanced class weights and random_state=42. UMAP uses random initialization, an adaptive neighbor count and an explicit compatibility layout. PCA needs at least two samples and two nonconstant predictors; model/embedding sample-size requirements still apply.

These are **exploratory analyses of the fitted dataset**. This notebook does not implement held-out or animal-level cross-validation and does not establish predictive generalization from separation in an embedding.

The module separately exports:

- model_feature_matrix.csv: the numeric matrix actually used.
- analysis_metadata.json: descriptor definition, predictor names, sample size, versions and model settings.
- rf_importance_table.csv: Random Forest impurity-based importance.
- shap_importance_table.csv: mean absolute SHAP importance from that same fitted classifier and predictor matrix.

RF importance and mean absolute SHAP are different quantities and their rankings need not agree. For multiclass SHAP, the table averages absolute contributions over samples and classes. The summary visualization uses class-averaged absolute contributions per sample/feature, not signed class-specific effects; its horizontal values cannot be read as positive/negative effects on a particular class. The two-dimensional fallback retains the SHAP values returned by the explainer.

All figures and the SHAP table are regenerated together. Changes to descriptors, exclusions, preprocessing or input images require refitting and re-exporting the corresponding results.


In [ ]:
def prepare_model_features(df, exclude=('cell_id', 'image_id')):
    """Drop legacy duplicates and unusable features before PCA/UMAP/RF/SHAP."""
    legacy = {'total_length', 'bifurcations', 'bifurcation_density', 'branching_index'}
    columns = [c for c in df.select_dtypes(include=[np.number]).columns
               if c not in set(exclude) | legacy and not c.startswith('mfs_scale_')]
    result = df[columns].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(axis=1, how='all')
    result = result.loc[:, result.nunique(dropna=True) > 1]
    return result.fillna(result.median())


# ============================================================
# Dimensionality Reduction & Feature Importance
# ============================================================

print("\n[INFO] ===== Module 10 - Dimensionality Reduction & Feature Importance =====")

# ------------------------------------------------------------
# 1. Auto-detect master dataframe
# ------------------------------------------------------------
possible_master_names = ["df_master", "df_all", "df_features", "df_ordered"]

df_ml = None
for name in possible_master_names:
    if name in globals():
        df_ml = globals()[name].copy()
        break

if df_ml is None:
    raise NameError("No master dataframe found. Please create df_master.")

# ------------------------------------------------------------
# 2. Auto-detect grouping column
# ------------------------------------------------------------
possible_group_cols = ["cell_state", "condition", "group", "stage", "label"]

GROUP_COL = None
for col in possible_group_cols:
    if col in df_ml.columns:
        GROUP_COL = col
        break

if GROUP_COL is None:
    raise KeyError("No grouping column found.")

# ------------------------------------------------------------
# 3. Select numeric morphometric metrics
# ------------------------------------------------------------
EXCLUDE_COLS = ["cell_id", "image_id"]

numeric_cols = [
    c for c in df_ml.select_dtypes(include=[np.number]).columns
    if c not in EXCLUDE_COLS
]

X_df = prepare_model_features(df_ml, exclude=EXCLUDE_COLS)
numeric_cols = X_df.columns.tolist()
X_df.to_csv(os.path.join(EXPORT_DIR, "Module 10 - model_feature_matrix.csv"), index=False)
metadata = {"analysis_schema": "2.0", "model": "RandomForestClassifier",
            "random_state": 42, "n_estimators": 500,
            "feature_names": numeric_cols, "n_samples": len(X_df),
            "units": "pixels of the preprocessed image",
            "descriptor_name": "Multiscale Fractal Spectrum (MFS)",
            "descriptor_definition": "Scale-dependent profile of local box-counting slopes: -d log N(s) / d log s.",
            "descriptor_notes": "Spectrum denotes variation across spatial scales, not a conventional multifractal singularity spectrum. Internal local_fd_scale_* columns identify the corrected slope calculation. total_length removed.",
            "versions": {name: package_version(name) for name in
                         ["numpy", "pandas", "pingouin", "scikit-learn", "shap", "scikit-image"]}}
with open(os.path.join(EXPORT_DIR, "Module 10 - analysis_metadata.json"), "w", encoding="utf-8") as handle:
    json.dump(metadata, handle, indent=2)
X = X_df.values
y = df_ml[GROUP_COL].values

# ------------------------------------------------------------
# 4. Standardize features
# ------------------------------------------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============================================================
# 10.1 PCA
# ============================================================

if min(X_scaled.shape) < 2:
    raise ValueError("PCA requires at least two samples and two nonconstant features.")
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_pca[:,0], y=X_pca[:,1],
    hue=y, palette="viridis", s=60, alpha=0.9
)
plt.title("PCA — PC1 vs PC2")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.legend(title=GROUP_COL)
plt.tight_layout()

plt.savefig(os.path.join(EXPORT_DIR, "Module 10 - pca.png"), dpi=600, bbox_inches="tight")
plt.show()

# PCA loadings
pca_loadings = pd.DataFrame(
    pca.components_.T,
    index=numeric_cols,
    columns=["PC1", "PC2"]
).sort_values("PC1", ascending=False)

# ============================================================
# 10.2 UMAP
# ============================================================

umap_model = umap.UMAP(
    n_neighbors=min(15, max(2, len(X_scaled) - 1)),
    compatibility_layout=True,
    init="random",
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

X_umap = umap_model.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
sns.scatterplot(
    x=X_umap[:,0], y=X_umap[:,1],
    hue=y, palette="viridis", s=60, alpha=0.9
)
plt.title("UMAP Embedding")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.legend(title=GROUP_COL)
plt.tight_layout()

plt.savefig(os.path.join(EXPORT_DIR, "Module 10 - umap.png"), dpi=600, bbox_inches="tight")
plt.show()

# ============================================================
# 10.3 Random Forest Feature Importance
# ============================================================

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced"
)
rf.fit(X_scaled, y)

importances = pd.DataFrame({
    "metric": numeric_cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

importances.to_csv(os.path.join(EXPORT_DIR, "Module 10 - rf_importance_table.csv"), index=False)

plt.figure(figsize=(8,10))
top_importances = importances.head(20)

sns.barplot(
    data=top_importances,
    x="importance",
    y="metric",
    hue="metric",
    palette="viridis",
    dodge=False,
    legend=False
)

plt.title("Top 20 Most Important Features (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Metric")
plt.tight_layout()

plt.savefig(os.path.join(EXPORT_DIR, "Module 10 - rf_top20.png"), dpi=600, bbox_inches="tight")
plt.show()

# ============================================================
# SHAP Interpretability
# ============================================================

# SHAP warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Use original feature names
feature_names = X_df.columns.tolist()

# SHAP explainer for tree-based models
explainer = shap.TreeExplainer(rf)

# Compute SHAP values
shap_values = explainer.shap_values(X_scaled)

# ------------------------------------------------------------
# Handle multiclass outputs robustly
# ------------------------------------------------------------

if isinstance(shap_values, list):
    # Convert to array: [n_classes, n_samples, n_features]
    shap_array = np.array(shap_values)

    # Mean absolute SHAP across classes and samples
    mean_abs_shap = np.mean(np.abs(shap_array), axis=(0, 1))

    # For summary plot: class-averaged absolute contributions (not signed).
    shap_for_plot = np.mean(np.abs(shap_array), axis=0)

else:
    shap_array = np.array(shap_values)

    if shap_array.ndim == 3:
        # [n_samples, n_features, n_classes]
        mean_abs_shap = np.mean(np.abs(shap_array), axis=(0, 2))
        shap_for_plot = np.mean(np.abs(shap_array), axis=2)
    else:
        # binary/single-output fallback
        mean_abs_shap = np.mean(np.abs(shap_array), axis=0)
        shap_for_plot = shap_array

# ------------------------------------------------------------
# Create SHAP table
# ------------------------------------------------------------
df_shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

# ------------------------------------------------------------
# SHAP bar plot
# ------------------------------------------------------------
plt.figure(figsize=(8, 10))
top_shap = df_shap_importance.head(20).sort_values("mean_abs_shap", ascending=True)

plt.barh(top_shap["feature"], top_shap["mean_abs_shap"])
plt.xlabel("Mean |SHAP value|")
plt.ylabel("Feature")
plt.title("Top 20 Features by SHAP Importance")
plt.tight_layout()
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 10 - shap_top20_barplot.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# ------------------------------------------------------------
# SHAP summary plot
# ------------------------------------------------------------
shap.summary_plot(
    shap_for_plot,
    X_df,
    show=False
)
plt.tight_layout()
plt.savefig(
    os.path.join(EXPORT_DIR, "Module 10 - shap_summary_plot.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

**Figure 17. Exploratory embeddings, RF importance and SHAP summaries.**

The exported files show PCA, UMAP, RF impurity importance, the SHAP importance bar plot and the SHAP summary distribution. If assembled in that sequence, panels A/B are PCA/UMAP, C is RF importance, **D is the mean-absolute-SHAP bar plot and E is the SHAP summary**; the notebook exports individual images rather than enforcing a composite-panel layout.

The SHAP bar plot and Table 12 use the same mean absolute values from the same fitted model. They should be assembled from one execution. RF and SHAP rankings measure different notions of importance. Multiclass summary values are class-averaged absolute contributions, not signed effects. Embedding separation and in-sample importance are exploratory and are not independent validation of biological states or classifier performance.


In [ ]:
# ------------------------------------------------------------
# SHAP importance table
# ------------------------------------------------------------
df_shap_importance.to_csv(
    os.path.join(EXPORT_DIR, "Module 10 - shap_importance_table.csv"),
    index=False
)

print("\n[INFO] ===== Module 10 Completed =====")

print("\n[INFO] ===== Finished =====")

**Table 12. Mean absolute SHAP importance from the current fitted classifier.**

Each row identifies a predictor and its mean absolute SHAP contribution, averaged over samples and classes for multiclass output. The table and SHAP bar figure share the same calculated vector. This is distinct from RF impurity-based importance. Feature relevance is model- and dataset-dependent; correlated predictors can share attribution. Use only the table exported with the corresponding current figures and model feature matrix.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">


### Final Remarks — A Multiscale Morphometric Workflow

This notebook combines global mask complexity, spatial heterogeneity, grayscale texture, an operational MFS profile of local box-counting slopes, pixel-graph organization and geometric Sholl profiles. These descriptors provide complementary views of the analyzed images; none alone establishes a functional microglial state.

The synthetic generator supports demonstration and regression checks. Experimental conclusions require appropriate image quality, sampling, biological replication and statistical design. Version 1.1.0 corrects the descriptor definitions and inferential exports; results from version 1.0.0 must be recomputed before reuse in figures, tables or model interpretation.


<hr style="border: 3px solid #444; margin-top: 40px; margin-bottom: 40px;">

### Outlook — Extensions Requiring Separate Validation

Potential extensions include explicit soma-centred Sholl analysis, calibrated geometric branch lengths, clustering junction pixels into anatomical events, origin-averaged box-counting profiles, hierarchical models for cells nested in animals/patients, and held-out validation of classification.

A conventional multifractal spectrum based on mass probabilities and q-dependent scaling would be a separate method, requiring its own implementation and validation. It is not implied by retaining the operational Multiscale Fractal Spectrum (MFS) name used here.
